In [1]:
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
import sys

!"{sys.executable}" -m pip install awswrangler -q
!"{sys.executable}" -m pip install optbinning -q
!"{sys.executable}" -m pip install lightgbm -q
!"{sys.executable}" -m pip install xgboost --prefer-binary -q
!"{sys.executable}" -m pip install boto3 -q


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

In [4]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [ ]:
import pandas as pd

# ===============================================================
# 1. Parámetros de tu bucket y prefijo
# ===============================================================
bucket_name  = 'ibk-discovery-comercial-us-east-1-654654352211-data'
model_prefix = 'discovery/comercial/sanherna/PLAFT/PN/MASIVO/DATA_INFERENCIA_PILOTO'

# ===============================================================
# 2. Rutas en S3
# ===============================================================
base_s3_path = f"s3://{bucket_name}/{model_prefix}/data_dev_model"

headers_path     = f"{base_s3_path}/headers_total.csv"
train_path       = f"{base_s3_path}/train_total.csv"
val_path         = f"{base_s3_path}/validation_total.csv"
test_path        = f"{base_s3_path}/test_total.csv"
extras_val_path  = f"{base_s3_path}/extras_validation_total.csv"
extras_test_path = f"{base_s3_path}/extras_test_total.csv"

# ===============================================================
# 3. Cargar headers (orden y tipos)
# ===============================================================
headers      = pd.read_csv(headers_path)
column_order = headers['variables'].tolist()

print(f"Se cargaron {len(column_order)} variables desde S3")
print("Primeras 10 variables:", column_order[:10])

# ===============================================================
# 4. Cargar SOLO test (mes 202607)
#    train y val se omiten para ahorrar memoria
# ===============================================================
df_test = pd.read_csv(test_path, header=None, names=column_order)

print(f"\nShape test completo: {df_test.shape}")

# ===============================================================
# 5. Cargar extras y pegar columnas de reporting
# ===============================================================
extras_test = pd.read_csv(extras_test_path)

df_test = pd.concat(
    [df_test.reset_index(drop=True),
     extras_test[['cod_mes', 'key_value', 'cuc_num']]], axis=1
)

# ===============================================================
# 6. Filtrar solo cod_mes == '202607'
# ===============================================================
MES_FILTRO = '202607'

# cod_mes puede venir como int o string según el CSV → normalizar
df_test['cod_mes'] = df_test['cod_mes'].astype(str).str.strip()

df_test = df_test[df_test['cod_mes'] == MES_FILTRO].reset_index(drop=True)

# ===============================================================
# 7. Verificación final
# ===============================================================
print(f"\n✅ Filtro aplicado: cod_mes == '{MES_FILTRO}'")
print(f"   df_test shape  : {df_test.shape}")
print(f"   Target rate    : {df_test['target'].mean():.4f}" if 'target' in df_test.columns else "   (columna 'target' no disponible en este split)")

display(df_test[['key_value', 'cuc_num', 'cod_mes', 'target']].head() if 'target' in df_test.columns
        else df_test[['key_value', 'cuc_num', 'cod_mes']].head())


In [40]:
df_1 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202602.txt",
    sep='|' 
)

In [41]:
df_2 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202601.txt",
    sep='|' 
)

In [42]:
df_3 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202511.txt",
    sep='|' 
)

In [43]:
df_4 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202508.txt",
    sep='|' 
)

In [44]:
df_5 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202509.txt",
    sep='|' 
)

In [45]:
df_6 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202510.txt",
    sep='|' 
)

In [46]:
df_7 = s3_read_csv(
    f"s3://{bucket_name}/{model_prefix}/REPLICA_OUTPUT/scr_Plaft-PN-RentaAlta_202512.txt",
    sep='|' 
)

In [47]:
df_8= pd.merge(df_1, df_2, how='outer').merge(df_3, how='outer').merge(df_4, how='outer').merge(df, how='outer').merge(df_5, how='outer').merge(df_6, how='outer').merge(df_7, how='outer')

In [48]:
df_8.variable1.value_counts()

variable1
SIN_INFO           2027982
AUTOMATICA            3279
MANUAL                 715
SEMI AUTOMATICA        459
Name: count, dtype: int64

In [49]:
df_res = (
    df_8
    .groupby(['grupo_alerta', 'codmes'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

print(df_res)

codmes        202508  202509  202510  202511  202512  202601  202602  202603  \
grupo_alerta                                                                   
P4+P5         251993  253991  254803  262221  265370  268269  236402  236416   
P2               144     144     256      74      62      98     151     176   
P3               112      97     209      80      55      81     120     233   
P1                95      94     154      63      63      78     146     185   

codmes          TOTAL  
grupo_alerta           
P4+P5         2029465  
P2               1105  
P3                987  
P1                878  


In [50]:
import pandas as pd

# === Tu pivot original (cantidades) ===
df_res = (
    df_8
    .groupby(['grupo_alerta', 'codmes'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_alerta', columns='codmes', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

# === PORCENTAJES POR MES (suman 100% por columna / por mes) ===
df_months = df_res.iloc[:, :-1]                     # todas las columnas de meses (sin TOTAL)
month_totals = df_months.sum(axis=0)                # total por cada mes

df_pct = (df_months.div(month_totals, axis=1) * 100).round(1)
df_pct = df_pct.astype(str) + '%'

# Renombrar columnas para que quede claro
df_pct.columns = [f'{col} %' for col in df_pct.columns]

# === Mostrar resultados ===
print("=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===")
print(df_res)

print("\n=== PORCENTAJES por mes (cada columna suma 100%) ===")
print(df_pct)



=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===
codmes        202508  202509  202510  202511  202512  202601  202602  202603  \
grupo_alerta                                                                   
P4+P5         251993  253991  254803  262221  265370  268269  236402  236416   
P2               144     144     256      74      62      98     151     176   
P3               112      97     209      80      55      81     120     233   
P1                95      94     154      63      63      78     146     185   

codmes          TOTAL  
grupo_alerta           
P4+P5         2029465  
P2               1105  
P3                987  
P1                878  

=== PORCENTAJES por mes (cada columna suma 100%) ===
             202508 % 202509 % 202510 % 202511 % 202512 % 202601 % 202602 %  \
grupo_alerta                                                                  
P4+P5           99.9%    99.9%    99.8%    99.9%    99.9%    99.9%    99.8%   
P2               0.1%     0.1

In [51]:
df_final_alerta= df_8[(df_8.variable1 !='SIN_INFO')]

In [52]:
# Asignar 5 grupos (quintiles) por mes según score (grupo 1 = mayor score)
import numpy as np

required_cols = {'codmes', 'score'}
missing = required_cols - set(df_final_alerta.columns)
if missing:
    raise ValueError(f"df_final_alerta no tiene las columnas requeridas: {missing}")

# Copiamos para no modificar vista previa previa
_df = df_final_alerta.copy()
_df['codmes'] = _df['codmes'].astype(str)
_df['score_num'] = pd.to_numeric(_df['score'], errors='coerce')


def _asignar_quintiles_por_mes(sub_df: pd.DataFrame) -> pd.DataFrame:
    sub_df = sub_df.sort_values('score_num', ascending=False).reset_index(drop=True)
    n = len(sub_df)
    if n == 0:
        sub_df['grupo_score_quintil'] = np.nan
        return sub_df

    # Ranking 1 = mayor score
    sub_df['rank_score'] = np.arange(1, n + 1)

    if n >= 5:
        # qcut sobre el ranking asegura tamaños casi iguales y que rank 1 quede en grupo 1
        sub_df['grupo_score_quintil'] = pd.qcut(
            sub_df['rank_score'],
            q=5,
            labels=[1, 2, 3, 4, 5],
            duplicates='drop'
        )
    else:
        # Con menos de 5 filas, todo queda en grupo 1
        sub_df['grupo_score_quintil'] = 1

    sub_df['grupo_score_quintil'] = sub_df['grupo_score_quintil'].astype(int)
    return sub_df


df_final_alerta = (
    _df
    .groupby('codmes', group_keys=False)
    .apply(_asignar_quintiles_por_mes)
    .reset_index(drop=True)
)

# Resumen de control
resumen_quintil = (
    df_final_alerta
    .groupby(['codmes', 'grupo_score_quintil'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .sort_values(['codmes', 'grupo_score_quintil'])
)
print("Resumen de quintiles por mes (grupo 1 = mayor score):")
print(resumen_quintil.head(20))

Resumen de quintiles por mes (grupo 1 = mayor score):
    codmes  grupo_score_quintil  cantidad
0   202508                    1       106
1   202508                    2       106
2   202508                    3       106
3   202508                    4       106
4   202508                    5       106
5   202509                    1        95
6   202509                    2        95
7   202509                    3        94
8   202509                    4        95
9   202509                    5        95
10  202510                    1       189
11  202510                    2       188
12  202510                    3       188
13  202510                    4       188
14  202510                    5       188
15  202511                    1        72
16  202511                    2        72
17  202511                    3        72
18  202511                    4        72
19  202511                    5        72


In [53]:
df_final_alerta.head()

,codmes,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,score_num,rank_score,grupo_score_quintil
0,202508,C992DA398E2DEE3C2C3F8E6D7DF16A2416CAEC34A075CC...,12171100,plaft_pj_minorista,20260427,P1,0,0.999970,1,AUTOMATICA,NaN,NaN,0.999970,1,1
1,202508,BF6D32813EF90B079F9B374FA60E93C348853C00DCE3C0...,12441002,plaft_pj_minorista,20260427,P1,0,0.999932,2,MANUAL,NaN,NaN,0.999932,2,1
2,202508,FEC00783EA13C2B05640013E83F2054E3BD650AD0EB07D...,18035714,plaft_pj_minorista,20260427,P1,0,0.999920,3,AUTOMATICA,NaN,NaN,0.999920,3,1
3,202508,4FA583BD95BC0D793453F4A6C80FD1B397C058FB1AD3D6...,18134874,plaft_pj_minorista,20260427,P1,0,0.999896,4,AUTOMATICA,NaN,NaN,0.999896,4,1
4,202508,F4BB0A3E2E7524D037FE58C2739B74F865393FF1691B98...,9002382,plaft_pj_minorista,20260427,P1,0,0.999882,5,MANUAL,NaN,NaN,0.999882,5,1


In [54]:
#test_data_0825_alerta_1= test_data_0825_alerta[(test_data_0825_alerta.variable1 !=0)]

In [55]:
import pandas as pd

# === Tu pivot original (cantidades) ===
df_res = (
    df_final_alerta
    .groupby(['grupo_score_quintil', 'codmes'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_score_quintil', columns='codmes', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

# === PORCENTAJES POR MES (suman 100% por columna / por mes) ===
df_months = df_res.iloc[:, :-1]                     # todas las columnas de meses (sin TOTAL)
month_totals = df_months.sum(axis=0)                # total por cada mes

df_pct = (df_months.div(month_totals, axis=1) * 100).round(1)
df_pct = df_pct.astype(str) + '%'

# Renombrar columnas para que quede claro
df_pct.columns = [f'{col} %' for col in df_pct.columns]

# === Mostrar resultados ===
print("=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===")
print(df_res)

print("\n=== PORCENTAJES por mes (cada columna suma 100%) ===")
print(df_pct)

=== CANTIDADES ABSOLUTAS por grupo_alerta y codmes ===
codmes               202508  202509  202510  202511  202512  202601  202602  \
grupo_score_quintil                                                           
1                       106      95     189      72      76      90     132   
5                       106      95     188      72      76      90     132   
3                       106      94     188      72      76      89     132   
2                       106      95     188      72      75      89     132   
4                       106      95     188      72      75      89     132   

codmes               202603  TOTAL  
grupo_score_quintil                 
1                       133    893  
5                       133    892  
3                       133    890  
2                       132    889  
4                       132    889  

=== PORCENTAJES por mes (cada columna suma 100%) ===
                    202508 % 202509 % 202510 % 202511 % 202512 % 202601 %  \
g

In [56]:
df_final_alerta.head()


,codmes,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,score_num,rank_score,grupo_score_quintil
0,202508,C992DA398E2DEE3C2C3F8E6D7DF16A2416CAEC34A075CC...,12171100,plaft_pj_minorista,20260427,P1,0,0.999970,1,AUTOMATICA,NaN,NaN,0.999970,1,1
1,202508,BF6D32813EF90B079F9B374FA60E93C348853C00DCE3C0...,12441002,plaft_pj_minorista,20260427,P1,0,0.999932,2,MANUAL,NaN,NaN,0.999932,2,1
2,202508,FEC00783EA13C2B05640013E83F2054E3BD650AD0EB07D...,18035714,plaft_pj_minorista,20260427,P1,0,0.999920,3,AUTOMATICA,NaN,NaN,0.999920,3,1
3,202508,4FA583BD95BC0D793453F4A6C80FD1B397C058FB1AD3D6...,18134874,plaft_pj_minorista,20260427,P1,0,0.999896,4,AUTOMATICA,NaN,NaN,0.999896,4,1
4,202508,F4BB0A3E2E7524D037FE58C2739B74F865393FF1691B98...,9002382,plaft_pj_minorista,20260427,P1,0,0.999882,5,MANUAL,NaN,NaN,0.999882,5,1


In [57]:
%%time
query = """

WITH pd AS (
    SELECT DISTINCT
        -- Identificadores
        a.num_documento,
        a.codunico,
        date_format(
            date_parse(CAST(a.mes_base AS varchar), '%Y%m') + interval '1' month,
            '%Y%m'
        ) AS codmes_lag1,
        CAST(a.mes_base AS INTEGER) AS codmes
    

    FROM d_perm_aws.ds_alertplaft_mdl a
    WHERE a.mes_base between '202507' and  '202604' 
      AND a.subsegmento = 'Renta Alta'
),

target AS (
    SELECT 
        codunico,
        periodo_alerta,
        tipo_alerta_n2,
        MAX(calificacion_monitoreo) AS flg_alerta
    FROM e_perm_aws.t_alertas_plaft
    GROUP BY codunico, periodo_alerta, tipo_alerta_n2
)

SELECT 
    a.*,
    b.tipo_alerta_n2,
    CASE 
        WHEN b.flg_alerta = '1' THEN 1 
        ELSE 0 
    END AS target_m
FROM pd a
LEFT JOIN target b
    ON a.codunico = b.codunico
 AND cast(a.codmes as varchar) = b.periodo_alerta
-- AND codmes_lag1 = b.periodo_alerta
;"""

df_dataset = athena_query(query, database='disc_comercial')
df_dataset.head()

CPU times: total: 11 s
Wall time: 2min 9s


,num_documento,codunico,codmes_lag1,codmes,tipo_alerta_n2,target_m
0,6EDA2494670D3A592309001BE64A607413C988509D2BAB...,0013772436,202601,202512,<NA>,0
1,167749A0E9832BD1FCE8E24678919668E04FF13EF15E90...,0013800051,202509,202508,<NA>,0
2,9636D401C3ABAB8AC99749CD6A8BEF4481A64FB4130D3F...,0014453435,202509,202508,<NA>,0
3,08979F09F5929600938D936AC811EE0DD77CBBC2934BE1...,0012186898,202509,202508,<NA>,0
4,9626F6F04584E937C60AE4E9017967788754F1FAA1C275...,0009509460,202509,202508,<NA>,0


In [58]:
import pandas as pd

# Aseguramos tipos consistentes
df_final_alerta['codmes'] = df_final_alerta['codmes'].astype(str)
df_dataset['codmes'] = df_dataset['codmes'].astype(str)

df_final_alerta['codunico'] = pd.to_numeric(df_final_alerta['codunico'], errors='coerce').astype('Int64')
df_dataset['codunico'] = pd.to_numeric(df_dataset['codunico'], errors='coerce').astype('Int64')

# 1. Cross join parcial (o merge sin condición de mes primero)
df_4_temp = pd.merge(
    df_final_alerta,
    df_dataset,
    on=['num_documento', 'codunico'],
    how='left',
    suffixes=('_final', '_dataset')
)

# 2. Filtramos solo las filas donde codmes_dataset >= codmes_final
df_6 = df_4_temp[
    df_4_temp['codmes_dataset'] >= df_4_temp['codmes_final']
].copy()

# 3. (Opcional) Si hay múltiples matches por cliente-mes, puedes quedarte con el primero o el más cercano
# Ejemplo: ordenar por diferencia de mes y quedarte con el menor (más cercano al actual)
df_6['mes_diff'] = pd.to_numeric(df_6['codmes_dataset']) - pd.to_numeric(df_6['codmes_final'])

# Ordenar por cliente + mes_diff ascendente (el más cercano primero)
df_6 = df_6.sort_values(['num_documento', 'codunico', 'mes_diff'])

# Opcional: quedarte solo con el match más cercano por cada fila de df_final
df_6 = df_6.drop_duplicates(subset=['num_documento', 'codunico', 'codmes_final'], keep='first')

# Limpiar columnas innecesarias si quieres
df_6 = df_6.drop(columns=['mes_diff'], errors='ignore')

In [59]:
df_6.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,score_num,rank_score,grupo_score_quintil,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
40906,202603,00065D9F4EF04E37C37B82836CFAE31ED55A168DB3B885...,10476423,plaft_pj_minorista,20260429,P4+P5,0,0.695961,25559,AUTOMATICA,NaN,NaN,0.695961,627,5,202604,202603,AUTOMATICA,0
7574,202509,000F67CAC1ED52C33A2527AFC3F96FF94AA9F84EA59C60...,15218337,plaft_pj_minorista,20260427,P3,0,0.991826,256,AUTOMATICA,NaN,NaN,0.991826,256,3,202510,202509,AUTOMATICA,0
16596,202510,0034F12F582FA597330C484BEC5AE17D4C6E40AA29FF3F...,5756036,plaft_pj_minorista,20260427,P4+P5,0,0.957429,1063,MANUAL,NaN,NaN,0.957429,715,4,202511,202510,MANUAL,0
25523,202512,0034F12F582FA597330C484BEC5AE17D4C6E40AA29FF3F...,5756036,plaft_pj_minorista,20260427,P4+P5,0,0.566155,40501,AUTOMATICA,NaN,NaN,0.566155,332,5,202601,202512,AUTOMATICA,0
39386,202603,0035F6CC498586D8C0D39F67F22C6A18FB9D0D6AAB9FF1...,20451425,plaft_pj_minorista,20260429,P3,0,0.984259,486,AUTOMATICA,NaN,NaN,0.984259,456,4,202604,202603,AUTOMATICA,0


In [60]:
df_6.target_m.value_counts()

target_m
0    4051
1     402
Name: count, dtype: Int64

In [61]:
df_6.variable1.value_counts()

variable1
AUTOMATICA         3279
MANUAL              715
SEMI AUTOMATICA     459
Name: count, dtype: int64

In [62]:
#Nuevas alertas 

In [63]:
df_6.head()

,codmes_final,num_documento,codunico,modelo,fec_replica,grupo_alerta,grupo_corte_nueva_alerta,score,orden,variable1,variable2,variable3,score_num,rank_score,grupo_score_quintil,codmes_lag1,codmes_dataset,tipo_alerta_n2,target_m
40906,202603,00065D9F4EF04E37C37B82836CFAE31ED55A168DB3B885...,10476423,plaft_pj_minorista,20260429,P4+P5,0,0.695961,25559,AUTOMATICA,NaN,NaN,0.695961,627,5,202604,202603,AUTOMATICA,0
7574,202509,000F67CAC1ED52C33A2527AFC3F96FF94AA9F84EA59C60...,15218337,plaft_pj_minorista,20260427,P3,0,0.991826,256,AUTOMATICA,NaN,NaN,0.991826,256,3,202510,202509,AUTOMATICA,0
16596,202510,0034F12F582FA597330C484BEC5AE17D4C6E40AA29FF3F...,5756036,plaft_pj_minorista,20260427,P4+P5,0,0.957429,1063,MANUAL,NaN,NaN,0.957429,715,4,202511,202510,MANUAL,0
25523,202512,0034F12F582FA597330C484BEC5AE17D4C6E40AA29FF3F...,5756036,plaft_pj_minorista,20260427,P4+P5,0,0.566155,40501,AUTOMATICA,NaN,NaN,0.566155,332,5,202601,202512,AUTOMATICA,0
39386,202603,0035F6CC498586D8C0D39F67F22C6A18FB9D0D6AAB9FF1...,20451425,plaft_pj_minorista,20260429,P3,0,0.984259,486,AUTOMATICA,NaN,NaN,0.984259,456,4,202604,202603,AUTOMATICA,0


In [64]:
df_res = (
    df_6
    .groupby(['grupo_score_quintil', 'codmes_final'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_score_quintil', columns='codmes_final', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

print(df_res)

codmes_final         202508  202509  202510  202511  202512  202601  202602  \
grupo_score_quintil                                                           
1                       106      95     189      72      76      90     132   
5                       106      95     188      72      76      90     132   
3                       106      94     188      72      76      89     132   
2                       106      95     188      72      75      89     132   
4                       106      95     188      72      75      89     132   

codmes_final         202603  TOTAL  
grupo_score_quintil                 
1                       133    893  
5                       133    892  
3                       133    890  
2                       132    889  
4                       132    889  


In [65]:
df_7= df_6[(df_6.variable1 !='SIN_INFO')]

In [66]:
df_res = (
    df_7
    .groupby(['grupo_score_quintil', 'codmes_final'], as_index=False)
    .size()
    .rename(columns={'size': 'cantidad'})
    .pivot(index='grupo_score_quintil', columns='codmes_final', values='cantidad')
    .fillna(0)
    .astype(int)
)

df_res['TOTAL'] = df_res.sum(axis=1)
df_res = df_res.sort_values('TOTAL', ascending=False)

print(df_res)

codmes_final         202508  202509  202510  202511  202512  202601  202602  \
grupo_score_quintil                                                           
1                       106      95     189      72      76      90     132   
5                       106      95     188      72      76      90     132   
3                       106      94     188      72      76      89     132   
2                       106      95     188      72      75      89     132   
4                       106      95     188      72      75      89     132   

codmes_final         202603  TOTAL  
grupo_score_quintil                 
1                       133    893  
5                       133    892  
3                       133    890  
2                       132    889  
4                       132    889  


In [67]:
# Resumen por grupo_alerta: total de casos y casos positivos (target_m = 1)
if 'target_m' not in df_7.columns:
    raise ValueError("df_7 no tiene la columna target_m")

resumen_grupo = (
    df_7.assign(target_m_num=pd.to_numeric(df_7['target_m'], errors='coerce').fillna(0).astype(int))
       .groupby('grupo_score_quintil', as_index=False)
       .agg(
           total_casos=('target_m_num', 'size'),
           casos_positivos=('target_m_num', 'sum')
       )
       .sort_values('grupo_score_quintil') 
       
)

resumen_grupo['tasa_positivos'] = (resumen_grupo['casos_positivos'] / resumen_grupo['total_casos']).round(4)

print("=== Casos por grupo_alerta ===")
display(resumen_grupo)

=== Casos por grupo_alerta ===


,grupo_score_quintil,total_casos,casos_positivos,tasa_positivos
0,1,893,159,0.1781
1,2,889,98,0.1102
2,3,890,75,0.0843
3,4,889,41,0.0461
4,5,892,29,0.0325


In [68]:
# Resumen mensual por mes (codmes_final) y grupo_score_quintil
column_mes = 'codmes_final'
required_cols = {column_mes, 'grupo_score_quintil', 'target_m'}
missing_cols = required_cols - set(df_7.columns)
if missing_cols:
    raise ValueError(f"df_7 no tiene las columnas requeridas: {missing_cols}")

resumen_mensual = (
    df_7.assign(
        codmes_final=df_7[column_mes].fillna('SIN_MES').astype(str),
        target_m_num=pd.to_numeric(df_7['target_m'], errors='coerce').fillna(0).astype(int)
    )
    .groupby(['codmes_final', 'grupo_score_quintil'], as_index=False)
    .agg(
        total_casos=('target_m_num', 'size'),
        casos_positivos=('target_m_num', 'sum')
    )
    .sort_values(['codmes_final', 'grupo_score_quintil'])
)

resumen_mensual['tasa_positivos'] = (
    resumen_mensual['casos_positivos'] / resumen_mensual['total_casos']
).round(4)

print("=== Resumen mensual por codmes_final y grupo_score_quintil ===")
display(resumen_mensual)

pivot_mensual = (
    resumen_mensual
    .pivot_table(
        index='grupo_score_quintil',
        columns='codmes_final',
        values='total_casos',
        fill_value=0,
        aggfunc='sum'
    )
    .astype(int)
)

pivot_mensual['TOTAL'] = pivot_mensual.sum(axis=1)
pivot_mensual = pivot_mensual.sort_values('TOTAL', ascending=False)

print("=== Total de casos por grupo_score_quintil (columnas = codmes_final) ===")
display(pivot_mensual)


=== Resumen mensual por codmes_final y grupo_score_quintil ===


,codmes_final,grupo_score_quintil,total_casos,casos_positivos,tasa_positivos
0,202508,1,106,25,0.2358
1,202508,2,106,20,0.1887
2,202508,3,106,21,0.1981
3,202508,4,106,6,0.0566
4,202508,5,106,3,0.0283
5,202509,1,95,14,0.1474
6,202509,2,95,9,0.0947
7,202509,3,94,4,0.0426
8,202509,4,95,2,0.0211
9,202509,5,95,5,0.0526


=== Total de casos por grupo_score_quintil (columnas = codmes_final) ===


codmes_final,202508,202509,202510,202511,202512,202601,202602,202603,TOTAL
grupo_score_quintil,,,,,,,,,
1,106,95,189,72,76,90,132,133,893
5,106,95,188,72,76,90,132,133,892
3,106,94,188,72,76,89,132,133,890
2,106,95,188,72,75,89,132,132,889
4,106,95,188,72,75,89,132,132,889


In [69]:
# Apertura mensual por tipo de alerta dentro de cada grupo_score_quintil
column_tipo = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else 'tipo_alerta'
required_cols_tipo = {column_mes, 'grupo_score_quintil', 'target_m', column_tipo}
missing_cols_tipo = required_cols_tipo - set(df_7.columns)
if missing_cols_tipo:
    raise ValueError(f"df_7 no tiene las columnas requeridas: {missing_cols_tipo}")

resumen_mensual_tipo = (
    df_7.assign(
        codmes_final=df_7[column_mes].fillna('SIN_MES').astype(str),
        tipo_alerta=df_7[column_tipo].fillna('SIN_TIPO_ALERTA').astype(str),
        target_m_num=pd.to_numeric(df_7['target_m'], errors='coerce').fillna(0).astype(int)
    )
    .groupby(['codmes_final', 'grupo_score_quintil', 'tipo_alerta'], as_index=False)
    .agg(
        total_casos=('target_m_num', 'size'),
        casos_positivos=('target_m_num', 'sum')
    )
    .sort_values(['codmes_final', 'grupo_score_quintil', 'tipo_alerta'])
)

resumen_mensual_tipo['tasa_positivos'] = (
    resumen_mensual_tipo['casos_positivos'] / resumen_mensual_tipo['total_casos']
).round(4)

print("=== Resumen mensual por codmes_final, grupo_score_quintil y tipo de alerta ===")
display(resumen_mensual_tipo)

resumen_tipo_mes = (
    resumen_mensual_tipo
    .groupby(['codmes_final', 'tipo_alerta'], as_index=False)
    .agg(
        total_casos=('total_casos', 'sum'),
        casos_positivos=('casos_positivos', 'sum')
    )
    .sort_values(['codmes_final', 'total_casos'], ascending=[True, False])
)

resumen_tipo_mes['tasa_positivos'] = (
    resumen_tipo_mes['casos_positivos'] / resumen_tipo_mes['total_casos']
).round(4)

print("=== Resumen mensual agregado por tipo de alerta ===")
display(resumen_tipo_mes)

pivot_tipo = (
    resumen_tipo_mes
    .pivot_table(
        index='tipo_alerta',
        columns='codmes_final',
        values='total_casos',
        aggfunc='sum',
        fill_value=0
    )
    .astype(int)
)

pivot_tipo['TOTAL'] = pivot_tipo.sum(axis=1)
pivot_tipo = pivot_tipo.sort_values('TOTAL', ascending=False)

print("=== Total de casos por tipo de alerta (columnas = codmes_final) ===")
display(pivot_tipo)


=== Resumen mensual por codmes_final, grupo_score_quintil y tipo de alerta ===


,codmes_final,grupo_score_quintil,tipo_alerta,total_casos,casos_positivos,tasa_positivos
0,202508,1,AUTOMATICA,62,6,0.0968
1,202508,1,MANUAL,24,13,0.5417
2,202508,1,SEMI AUTOMATICA,20,6,0.3000
3,202508,2,AUTOMATICA,78,7,0.0897
4,202508,2,MANUAL,15,9,0.6000
5,202508,2,SEMI AUTOMATICA,13,4,0.3077
6,202508,3,AUTOMATICA,80,7,0.0875
7,202508,3,MANUAL,13,12,0.9231
8,202508,3,SEMI AUTOMATICA,13,2,0.1538
9,202508,4,AUTOMATICA,95,3,0.0316


=== Resumen mensual agregado por tipo de alerta ===


,codmes_final,tipo_alerta,total_casos,casos_positivos,tasa_positivos
0,202508,AUTOMATICA,411,24,0.0584
2,202508,SEMI AUTOMATICA,62,12,0.1935
1,202508,MANUAL,57,39,0.6842
3,202509,AUTOMATICA,374,19,0.0508
5,202509,SEMI AUTOMATICA,62,11,0.1774
4,202509,MANUAL,38,4,0.1053
7,202510,MANUAL,497,10,0.0201
6,202510,AUTOMATICA,378,16,0.0423
8,202510,SEMI AUTOMATICA,66,10,0.1515
9,202511,AUTOMATICA,312,9,0.0288


=== Total de casos por tipo de alerta (columnas = codmes_final) ===


codmes_final,202508,202509,202510,202511,202512,202601,202602,202603,TOTAL
tipo_alerta,,,,,,,,,
AUTOMATICA,411,374,378,312,345,383,527,551,3281
MANUAL,57,38,497,13,2,32,30,45,714
SEMI AUTOMATICA,62,62,66,35,31,32,103,67,458


In [70]:
# -- Bloque adicional: sólo tipo_alerta == 'AUTOMATICA' --
column_tipo_auto = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else 'tipo_alerta'
required_cols_auto = {column_mes, 'grupo_score_quintil', 'target_m', column_tipo_auto}
missing_cols_auto = required_cols_auto - set(df_7.columns)
if missing_cols_auto:
    raise ValueError(f"df_7 no tiene las columnas requeridas para AUTOMATICA: {missing_cols_auto}")

# Filtrar universo AUTOMATICA
df_auto = df_7[df_7[column_tipo_auto] == 'AUTOMATICA'].copy()
if df_auto.empty:
    print("No hay registros con tipo_alerta = 'AUTOMATICA' en df_7")
else:
    df_auto['codmes_final'] = df_auto[column_mes].fillna('SIN_MES').astype(str)
    df_auto['target_m_num'] = pd.to_numeric(df_auto['target_m'], errors='coerce').fillna(0).astype(int)

    resumen_mensual_auto = (
        df_auto
        .groupby(['codmes_final', 'grupo_score_quintil'], as_index=False)
        .agg(
            total_casos=('target_m_num', 'size'),
            casos_positivos=('target_m_num', 'sum')
        )
        .sort_values(['codmes_final', 'grupo_score_quintil'])
    )
    resumen_mensual_auto['tasa_positivos'] = (
        resumen_mensual_auto['casos_positivos'] / resumen_mensual_auto['total_casos']
    ).round(4)

    print("=== Resumen mensual SOLO AUTOMATICA por codmes_final y grupo_score_quintil ===")
    display(resumen_mensual_auto)

    resumen_auto_mes = (
        df_auto
        .groupby(['codmes_final'], as_index=False)
        .agg(
            total_casos=('target_m_num', 'size'),
            casos_positivos=('target_m_num', 'sum')
        )
        .sort_values(['codmes_final', 'total_casos'], ascending=[True, False])
    )
    resumen_auto_mes['tasa_positivos'] = (
        resumen_auto_mes['casos_positivos'] / resumen_auto_mes['total_casos']
    ).round(4)

    print("=== Resumen mensual agregado SOLO AUTOMATICA ===")
    display(resumen_auto_mes)

    pivot_auto = (
        resumen_mensual_auto
        .pivot_table(
            index='grupo_score_quintil',
            columns='codmes_final',
            values='total_casos',
            aggfunc='sum',
            fill_value=0
        )
        .astype(int)
    )
    pivot_auto['TOTAL'] = pivot_auto.sum(axis=1)
    pivot_auto = pivot_auto.sort_values('TOTAL', ascending=False)

    print("=== Total de casos SOLO AUTOMATICA por grupo_score_quintil (columnas = codmes_final) ===")
    display(pivot_auto)

=== Resumen mensual SOLO AUTOMATICA por codmes_final y grupo_score_quintil ===


,codmes_final,grupo_score_quintil,total_casos,casos_positivos,tasa_positivos
0,202508,1,62,6,0.0968
1,202508,2,78,7,0.0897
2,202508,3,80,7,0.0875
3,202508,4,95,3,0.0316
4,202508,5,96,1,0.0104
5,202509,1,61,8,0.1311
6,202509,2,69,5,0.0725
7,202509,3,76,2,0.0263
8,202509,4,87,1,0.0115
9,202509,5,81,3,0.0370


=== Resumen mensual agregado SOLO AUTOMATICA ===


,codmes_final,total_casos,casos_positivos,tasa_positivos
0,202508,411,24,0.0584
1,202509,374,19,0.0508
2,202510,378,16,0.0423
3,202511,312,9,0.0288
4,202512,345,3,0.0087
5,202601,383,27,0.0705
6,202602,527,44,0.0835
7,202603,551,62,0.1125


=== Total de casos SOLO AUTOMATICA por grupo_score_quintil (columnas = codmes_final) ===


codmes_final,202508,202509,202510,202511,202512,202601,202602,202603,TOTAL
grupo_score_quintil,,,,,,,,,
5,96,81,81,67,75,82,117,128,727
4,95,87,80,71,73,79,109,116,710
3,80,76,76,62,72,80,111,112,669
2,78,69,82,61,68,77,100,102,637
1,62,61,59,51,57,65,90,93,538


In [71]:
# Resumen mensual por mes (codmes_final) y grupo_score_quintil
column_mes = 'codmes_final'
required_cols = {column_mes, 'grupo_score_quintil', 'target_m'}
missing_cols = required_cols - set(df_7.columns)
if missing_cols:
    raise ValueError(f"df_7 no tiene las columnas requeridas: {missing_cols}")

resumen_mensual = (
    df_7.assign(
        codmes_final=df_7[column_mes].fillna('SIN_MES').astype(str),
        target_m_num=pd.to_numeric(df_7['target_m'], errors='coerce').fillna(0).astype(int)
    )
    .groupby(['codmes_final', 'grupo_score_quintil'], as_index=False)
    .agg(
        total_casos=('target_m_num', 'size'),
        casos_positivos=('target_m_num', 'sum')
    )
    .sort_values(['codmes_final', 'grupo_score_quintil'])
)

resumen_mensual['tasa_positivos'] = (
    resumen_mensual['casos_positivos'] / resumen_mensual['total_casos']
).round(4)

print("=== Resumen mensual por codmes_final y grupo_score_quintil ===")
display(resumen_mensual)

pivot_mensual = (
    resumen_mensual
    .pivot_table(
        index='grupo_score_quintil',
        columns='codmes_final',
        values='total_casos',
        fill_value=0,
        aggfunc='sum'
    )
    .astype(int)
)

pivot_mensual['TOTAL'] = pivot_mensual.sum(axis=1)
pivot_mensual = pivot_mensual.sort_values('TOTAL', ascending=False)

print("=== Total de casos por grupo_score_quintil (columnas = codmes_final) ===")
display(pivot_mensual)

=== Resumen mensual por codmes_final y grupo_score_quintil ===


,codmes_final,grupo_score_quintil,total_casos,casos_positivos,tasa_positivos
0,202508,1,106,25,0.2358
1,202508,2,106,20,0.1887
2,202508,3,106,21,0.1981
3,202508,4,106,6,0.0566
4,202508,5,106,3,0.0283
5,202509,1,95,14,0.1474
6,202509,2,95,9,0.0947
7,202509,3,94,4,0.0426
8,202509,4,95,2,0.0211
9,202509,5,95,5,0.0526


=== Total de casos por grupo_score_quintil (columnas = codmes_final) ===


codmes_final,202508,202509,202510,202511,202512,202601,202602,202603,TOTAL
grupo_score_quintil,,,,,,,,,
1,106,95,189,72,76,90,132,133,893
5,106,95,188,72,76,90,132,133,892
3,106,94,188,72,76,89,132,133,890
2,106,95,188,72,75,89,132,132,889
4,106,95,188,72,75,89,132,132,889


In [72]:
!pip install plotly

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [76]:
from pathlib import Path
import json

# --- Mapeo de quintiles a grupos visuales (sin agrupar P4+P5) ---
group_labels = {1: 'P1', 2: 'P2', 3: 'P3', 4: 'P4', 5: 'P5'}
group_colors = {'P1': '#00BE50', 'P2': '#5A5A5A', 'P3': '#7A7A7A', 'P4': '#ABABAB', 'P5': '#D4D4D4'}
groups_ordered = ['P1', 'P2', 'P3', 'P4', 'P5']

# --- Filtra periodos: excluye 202512 y 202510 ---
periodos_excluir = {'202512', '202510'}
rm = resumen_mensual.copy()
rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
rm['grupo_label'] = rm['grupo_score_quintil'].map(group_labels)

# Agrega por grupo_label (cada quintil es su propio grupo)
rm_agg = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(
        total_casos=('total_casos', 'sum'),
        casos_positivos=('casos_positivos', 'sum')
    )
)
rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)

periodos = sorted(rm_agg['codmes_final'].unique())

# --- Calcula los arrays para Chart.js ---
# precision_pct: tasa_positivos * 100 por quintil/mes (para grafico de barras "Precision")
# efec: tasa positivos por grupo - grafico de lineas
# riesgos_pct: % participacion en casos positivos
precision_pct = {}
efec = {}
riesgos_pct = {}
alertas_pct = {}  # se mantiene por compatibilidad

for grp in groups_ordered:
    p_vals, e_vals, r_vals, a_vals = [], [], [], []
    for per in periodos:
        sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
        total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
        total_pos_mes = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()

        if sub.empty:
            p_vals.append(0); e_vals.append(0); r_vals.append(0); a_vals.append(0)
        else:
            tc = sub['total_casos'].values[0]
            cp = sub['casos_positivos'].values[0]
            tp = sub['tasa_positivos'].values[0]
            p_vals.append(round(100 * tp, 1))                                          # Precision = tasa positivos %
            e_vals.append(round(100 * tp, 1))                                          # Tasa positivos (identico)
            r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)  # % Participacion positivos
            a_vals.append(round(100 * tc / total_casos_mes, 1) if total_casos_mes else 0)

    precision_pct[grp] = p_vals
    efec[grp] = e_vals
    riesgos_pct[grp] = r_vals
    alertas_pct[grp] = a_vals

# --- Tabla pivot con casos positivos ---
pivot_tc = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
)

pt_casos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos', fill_value=0, aggfunc='sum').astype(int)
pt_casos = pt_casos.reindex(groups_ordered)
pt_casos['TOTAL'] = pt_casos.sum(axis=1)

pt_pos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
pt_pos = pt_pos.reindex(groups_ordered)
pt_pos['TOTAL'] = pt_pos.sum(axis=1)

meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

table_header = """<thead>
  <tr>
    <th rowspan="2" style="vertical-align:middle;">Grupo</th>
    """ + ''.join(f'<th colspan="2" style="text-align:center;">{c}</th>' for c in meses_cols) + """
    <th colspan="2" style="text-align:center;">TOTAL</th>
  </tr>
  <tr>
    """ + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols) + """
    <th>Casos</th><th>Positivos</th>
  </tr>
</thead>"""

table_rows = ''
for i, grp in enumerate(groups_ordered):
    bg = '#f9fbfb' if i % 2 == 0 else '#fff'
    td_grp = f'<td style="font-weight:700;padding:10px 14px;background:{bg}"><span style="display:inline-block;width:12px;height:12px;border-radius:3px;background:{group_colors[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>'
    tds = ''
    for c in meses_cols:
        casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
        pos = pt_pos.loc[grp, c] if grp in pt_pos.index and c in pt_pos.columns else 0
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
    total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
    total_p = pt_pos.loc[grp, 'TOTAL'] if grp in pt_pos.index else 0
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
    table_rows += f'<tr>{td_grp}{tds}</tr>'

tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
for c in meses_cols:
    tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
    tp_col = int(pt_pos[c].sum()) if c in pt_pos.columns else 0
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
total_all_c = int(pt_casos['TOTAL'].sum())
total_all_p = int(pt_pos['TOTAL'].sum())
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{total_all_c:,}</td>'
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{total_all_p:,}</td>'
tr_tot += '</tr>'

table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

# --- Top tipos de alerta (excluye 202512 y 202510) ---
if 'tipo_alerta_n2' in df_7.columns:
    tipo_col = 'tipo_alerta_n2'
else:
    tipo_col = 'tipo_alerta' if 'tipo_alerta' in df_7.columns else None

tipo_alerta_html = ''
if tipo_col:
    df_7_filt = df_7[~df_7['codmes_final'].astype(str).isin(periodos_excluir)]
    resumen_tipo = df_7_filt.groupby(tipo_col).agg(
        total_casos=('target_m', 'size'),
        casos_positivos=('target_m', lambda x: (pd.to_numeric(x, errors='coerce') == 1).sum())
    )
    resumen_tipo['tasa_positivos'] = (resumen_tipo['casos_positivos'] / resumen_tipo['total_casos'] * 100).round(1)
    resumen_tipo = resumen_tipo.sort_values('total_casos', ascending=False).head(5)
    tipo_alerta_html = '<table class="tipo-alerta-table" style="border-collapse:collapse;width:100%"><thead><tr><th>Tipo de alerta</th><th>Total casos</th><th>Casos positivos</th><th>Tasa positivos</th></tr></thead><tbody>'
    for idx, row in resumen_tipo.iterrows():
        tipo_alerta_html += (
            f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
            f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
        )
    tipo_alerta_html += '</tbody></table>'

# --- Datos para Chart.js ---
groups_stacked = ['P5', 'P4', 'P3', 'P2', 'P1']

chart_data = {
    'periodos': periodos,
    'groups': groups_stacked,
    'groups_line': groups_ordered,
    'precision_pct': precision_pct,
    'efectividad': efec,
    'riesgos_pct': riesgos_pct,
    'groupColors': group_colors
}

html_tpl = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .tipo-alerta-table{width:100%;border-collapse:collapse}
    .tipo-alerta-table th{background:rgb(31,69,146);color:#fff;padding:10px;text-align:left}
    .tipo-alerta-table td{padding:10px;border-bottom:1px solid #e2e8f0}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
    @media print{body{padding:0;background:white}.pagina{box-shadow:none;max-width:100%}}
    @media(max-width:1400px){.fila-unica{grid-template-columns:1fr}.grafico-card{min-height:450px}}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional</h1>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Precisi&#243;n por Quintil (Tasa Positivos)</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos (acumulado)</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Top tipos de alerta (total periodo)</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true,
      maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff',
          font: { weight: 'bold', size: 11 },
          formatter: function(value) { return value >= 5 ? value + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)',
          textStrokeWidth: 2
        }
      },
      scales: {
        x: { stacked: true, grid: { display: false } },
        y: { stacked: true, display: false }
      }
    };
  }

  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }

  // Grafico 1: BARRAS APILADAS - Precision por Quintil (tasa_positivos %)
  new Chart(document.getElementById('leadsChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('precision_pct') },
    options: stackedOptions()
  });

  // Grafico 2: LINEAS - % Tasa Positivos por Grupo
  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp,
      data: chartData.efectividad[grp],
      borderColor: groupColors[grp],
      backgroundColor: groupColors[grp],
      borderWidth: 3,
      tension: 0.3,
      pointRadius: 5,
      pointBackgroundColor: '#fff',
      pointBorderColor: groupColors[grp],
      pointBorderWidth: 2,
      datalabels: {
        align: 'top',
        offset: 6,
        color: groupColors[grp],
        font: { weight: 'bold' },
        formatter: function(value) { return value ? value + '%' : ''; }
      }
    };
  });

  new Chart(document.getElementById('efecChart'), {
    type: 'line',
    data: { labels: periodos, datasets: lineDatasets },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } }
    }
  });

  // Grafico 3: barras apiladas % Participacion Casos Positivos
  new Chart(document.getElementById('desemChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('riesgos_pct') },
    options: stackedOptions()
  });

  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

html_final = (
    html_tpl
    .replace('PERIODO_START', str(periodos[0]))
    .replace('PERIODO_END', str(periodos[-1]))
    .replace('TABLE_HTML', table_html)
    .replace('TIPO_ALERTA_HTML', tipo_alerta_html)
    .replace('CHART_DATA_JSON', json.dumps(chart_data))
)

out_html = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_mensual_quintil_ejecutivo.html")
out_html.write_text(html_final, encoding='utf-8')
print(f"HTML ejecutivo generado en: {out_html}")


HTML ejecutivo generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo.html


In [75]:
from pathlib import Path
import json

# ============================================================
# HTML EJECUTIVO — misma lógica pero P2+P3 agrupados
# ============================================================

# --- Mapeo: quintiles 2 y 3 → 'P2+P3' ---
def _map_grupo_agrupado(q):
    if q == 1:   return 'P1'
    if q in (2, 3): return 'P2+P3'
    if q == 4:   return 'P4'
    if q == 5:   return 'P5'
    return 'OTRO'

group_colors_ag = {
    'P1':    '#00BE50',
    'P2+P3': '#5A5A5A',
    'P4':    '#ABABAB',
    'P5':    '#D4D4D4',
}
groups_ordered_ag = ['P1', 'P2+P3', 'P4', 'P5']

periodos_excluir = {'202512', '202510'}

rm = resumen_mensual.copy()
rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
rm['grupo_label'] = rm['grupo_score_quintil'].map(_map_grupo_agrupado)

rm_agg = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(
        total_casos=('total_casos', 'sum'),
        casos_positivos=('casos_positivos', 'sum')
    )
)
rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)

periodos = sorted(rm_agg['codmes_final'].unique())

# --- Calcula arrays para Chart.js ---
precision_pct = {}
efec          = {}
riesgos_pct   = {}
alertas_pct   = {}

for grp in groups_ordered_ag:
    p_vals, e_vals, r_vals, a_vals = [], [], [], []
    for per in periodos:
        sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
        total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
        total_pos_mes   = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()
        if sub.empty:
            p_vals.append(0); e_vals.append(0); r_vals.append(0); a_vals.append(0)
        else:
            tc = sub['total_casos'].values[0]
            cp = sub['casos_positivos'].values[0]
            tp = sub['tasa_positivos'].values[0]
            p_vals.append(round(100 * tp, 1))
            e_vals.append(round(100 * tp, 1))
            r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)
            a_vals.append(round(100 * tc / total_casos_mes, 1) if total_casos_mes else 0)
    precision_pct[grp] = p_vals
    efec[grp]          = e_vals
    riesgos_pct[grp]   = r_vals
    alertas_pct[grp]   = a_vals

# --- Tabla pivot ---
pivot_tc = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
)

pt_casos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos',    fill_value=0, aggfunc='sum').astype(int)
pt_casos = pt_casos.reindex(groups_ordered_ag)
pt_casos['TOTAL'] = pt_casos.sum(axis=1)

pt_pos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
pt_pos = pt_pos.reindex(groups_ordered_ag)
pt_pos['TOTAL'] = pt_pos.sum(axis=1)

meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

table_header = (
    """<thead><tr><th rowspan="2" style="vertical-align:middle;">Grupo</th>"""
    + ''.join(f'<th colspan="2" style="text-align:center;">{c}</th>' for c in meses_cols)
    + """<th colspan="2" style="text-align:center;">TOTAL</th></tr><tr>"""
    + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols)
    + """<th>Casos</th><th>Positivos</th></tr></thead>"""
)

table_rows = ''
for i, grp in enumerate(groups_ordered_ag):
    bg = '#f9fbfb' if i % 2 == 0 else '#fff'
    td_grp = (
        f'<td style="font-weight:700;padding:10px 14px;background:{bg}">'
        f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
        f'background:{group_colors_ag[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>'
    )
    tds = ''
    for c in meses_cols:
        casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
        pos   = pt_pos.loc[grp, c]   if grp in pt_pos.index   and c in pt_pos.columns   else 0
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
    total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
    total_p = pt_pos.loc[grp,   'TOTAL'] if grp in pt_pos.index   else 0
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
    table_rows += f'<tr>{td_grp}{tds}</tr>'

tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
for c in meses_cols:
    tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
    tp_col = int(pt_pos[c].sum())   if c in pt_pos.columns   else 0
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
total_all_c = int(pt_casos['TOTAL'].sum())
total_all_p = int(pt_pos['TOTAL'].sum())
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{total_all_c:,}</td>'
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{total_all_p:,}</td>'
tr_tot += '</tr>'

table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

# --- Top tipos de alerta ---
tipo_col = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else ('tipo_alerta' if 'tipo_alerta' in df_7.columns else None)
tipo_alerta_html = ''
if tipo_col:
    df_7_filt = df_7[~df_7['codmes_final'].astype(str).isin(periodos_excluir)]
    resumen_tipo = df_7_filt.groupby(tipo_col).agg(
        total_casos=('target_m', 'size'),
        casos_positivos=('target_m', lambda x: (pd.to_numeric(x, errors='coerce') == 1).sum())
    )
    resumen_tipo['tasa_positivos'] = (resumen_tipo['casos_positivos'] / resumen_tipo['total_casos'] * 100).round(1)
    resumen_tipo = resumen_tipo.sort_values('total_casos', ascending=False).head(5)
    tipo_alerta_html = '<table class="tipo-alerta-table" style="border-collapse:collapse;width:100%"><thead><tr><th>Tipo de alerta</th><th>Total casos</th><th>Casos positivos</th><th>Tasa positivos</th></tr></thead><tbody>'
    for idx, row in resumen_tipo.iterrows():
        tipo_alerta_html += (
            f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
            f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
        )
    tipo_alerta_html += '</tbody></table>'

# --- Datos Chart.js ---
groups_stacked_ag = list(reversed(groups_ordered_ag))   # P5, P4, P2+P3, P1

chart_data = {
    'periodos':     periodos,
    'groups':       groups_stacked_ag,
    'groups_line':  groups_ordered_ag,
    'precision_pct': precision_pct,
    'efectividad':  efec,
    'riesgos_pct':  riesgos_pct,
    'groupColors':  group_colors_ag,
}

html_tpl = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo (P2+P3 agrupados) - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .badge-agrupado{background:#5A5A5A;color:#fff;padding:5px 14px;border-radius:20px;font-size:12px;font-weight:700;margin-left:12px;letter-spacing:.5px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .tipo-alerta-table{width:100%;border-collapse:collapse}
    .tipo-alerta-table th{background:rgb(31,69,146);color:#fff;padding:10px;text-align:left}
    .tipo-alerta-table td{padding:10px;border-bottom:1px solid #e2e8f0}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
    @media print{body{padding:0;background:white}.pagina{box-shadow:none;max-width:100%}}
    @media(max-width:1400px){.fila-unica{grid-template-columns:1fr}.grafico-card{min-height:450px}}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <div style="display:flex;align-items:center">
      <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional</h1>
      <span class="badge-agrupado">P2+P3 agrupados</span>
    </div>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Precisi&#243;n por Grupo (Tasa Positivos)</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Top tipos de alerta (total periodo)</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial &#183; Grupos: P1 | P2+P3 | P4 | P5</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff', font: { weight: 'bold', size: 11 },
          formatter: function(v) { return v >= 5 ? v + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)', textStrokeWidth: 2
        }
      },
      scales: { x: { stacked: true, grid: { display: false } }, y: { stacked: true, display: false } }
    };
  }

  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }

  new Chart(document.getElementById('leadsChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('precision_pct') },
    options: stackedOptions()
  });

  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp, data: chartData.efectividad[grp],
      borderColor: groupColors[grp], backgroundColor: groupColors[grp],
      borderWidth: 3, tension: 0.3, pointRadius: 5,
      pointBackgroundColor: '#fff', pointBorderColor: groupColors[grp], pointBorderWidth: 2,
      datalabels: {
        align: 'top', offset: 6, color: groupColors[grp], font: { weight: 'bold' },
        formatter: function(v) { return v ? v + '%' : ''; }
      }
    };
  });

  new Chart(document.getElementById('efecChart'), {
    type: 'line',
    data: { labels: periodos, datasets: lineDatasets },
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } }
    }
  });

  new Chart(document.getElementById('desemChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('riesgos_pct') },
    options: stackedOptions()
  });

  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

html_final = (
    html_tpl
    .replace('PERIODO_START', str(periodos[0]))
    .replace('PERIODO_END',   str(periodos[-1]))
    .replace('TABLE_HTML',        table_html)
    .replace('TIPO_ALERTA_HTML',  tipo_alerta_html)
    .replace('CHART_DATA_JSON',   json.dumps(chart_data))
)

out_html = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_mensual_quintil_ejecutivo_P2P3.html")
out_html.write_text(html_final, encoding='utf-8')
print(f"HTML con P2+P3 agrupados generado en: {out_html}")
print(f"Grupos usados: {groups_ordered_ag}")
print(f"Periodos incluidos: {periodos}")


HTML con P2+P3 agrupados generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo_P2P3.html
Grupos usados: ['P1', 'P2+P3', 'P4', 'P5']
Periodos incluidos: ['202508', '202509', '202511', '202601', '202602', '202603']


In [73]:

from pathlib import Path
import json

# ============================================================
# HTML EJECUTIVO — P2+P3 agrupados Y P4+P5 agrupados
# ============================================================

def _map_grupo_p2p3_p4p5(q):
    if q == 1:       return 'P1'
    if q in (2, 3):  return 'P2+P3'
    if q in (4, 5):  return 'P4+P5'
    return 'OTRO'

group_colors_ag2 = {
    'P1':    '#00BE50',
    'P2+P3': '#5A5A5A',
    'P4+P5': '#D4D4D4',
}
groups_ordered_ag2 = ['P1', 'P2+P3', 'P4+P5']

periodos_excluir = {'202512', '202510'}

rm = resumen_mensual.copy()
rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
rm['grupo_label'] = rm['grupo_score_quintil'].map(_map_grupo_p2p3_p4p5)

rm_agg = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(
        total_casos=('total_casos', 'sum'),
        casos_positivos=('casos_positivos', 'sum')
    )
)
rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)

periodos = sorted(rm_agg['codmes_final'].unique())

# --- Calcula arrays para Chart.js ---
precision_pct = {}
efec          = {}
riesgos_pct   = {}

for grp in groups_ordered_ag2:
    p_vals, e_vals, r_vals = [], [], []
    for per in periodos:
        sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
        total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
        total_pos_mes   = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()
        if sub.empty:
            p_vals.append(0); e_vals.append(0); r_vals.append(0)
        else:
            tc = sub['total_casos'].values[0]
            cp = sub['casos_positivos'].values[0]
            tp = sub['tasa_positivos'].values[0]
            p_vals.append(round(100 * tp, 1))
            e_vals.append(round(100 * tp, 1))
            r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)
    precision_pct[grp] = p_vals
    efec[grp]          = e_vals
    riesgos_pct[grp]   = r_vals

# --- Tabla pivot ---
pivot_tc = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
)
pt_casos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos',    fill_value=0, aggfunc='sum').astype(int)
pt_casos = pt_casos.reindex(groups_ordered_ag2)
pt_casos['TOTAL'] = pt_casos.sum(axis=1)

pt_pos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
pt_pos = pt_pos.reindex(groups_ordered_ag2)
pt_pos['TOTAL'] = pt_pos.sum(axis=1)

meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

table_header = (
    """<thead><tr><th rowspan="2" style="vertical-align:middle;">Grupo</th>"""
    + ''.join(f'<th colspan="2" style="text-align:center;">{c}</th>' for c in meses_cols)
    + """<th colspan="2" style="text-align:center;">TOTAL</th></tr><tr>"""
    + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols)
    + """<th>Casos</th><th>Positivos</th></tr></thead>"""
)

table_rows = ''
for i, grp in enumerate(groups_ordered_ag2):
    bg = '#f9fbfb' if i % 2 == 0 else '#fff'
    td_grp = (
        f'<td style="font-weight:700;padding:10px 14px;background:{bg}">'
        f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
        f'background:{group_colors_ag2[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>'
    )
    tds = ''
    for c in meses_cols:
        casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
        pos   = pt_pos.loc[grp, c]   if grp in pt_pos.index   and c in pt_pos.columns   else 0
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
    total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
    total_p = pt_pos.loc[grp,   'TOTAL'] if grp in pt_pos.index   else 0
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
    table_rows += f'<tr>{td_grp}{tds}</tr>'

tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
for c in meses_cols:
    tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
    tp_col = int(pt_pos[c].sum())   if c in pt_pos.columns   else 0
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
total_all_c = int(pt_casos['TOTAL'].sum())
total_all_p = int(pt_pos['TOTAL'].sum())
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{total_all_c:,}</td>'
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{total_all_p:,}</td>'
tr_tot += '</tr>'

table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

# --- Top tipos de alerta ---
tipo_col = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else ('tipo_alerta' if 'tipo_alerta' in df_7.columns else None)
tipo_alerta_html = ''
if tipo_col:
    df_7_filt = df_7[~df_7['codmes_final'].astype(str).isin(periodos_excluir)]
    resumen_tipo = df_7_filt.groupby(tipo_col).agg(
        total_casos=('target_m', 'size'),
        casos_positivos=('target_m', lambda x: (pd.to_numeric(x, errors='coerce') == 1).sum())
    )
    resumen_tipo['tasa_positivos'] = (resumen_tipo['casos_positivos'] / resumen_tipo['total_casos'] * 100).round(1)
    resumen_tipo = resumen_tipo.sort_values('total_casos', ascending=False).head(5)
    tipo_alerta_html = '<table class="tipo-alerta-table" style="border-collapse:collapse;width:100%"><thead><tr><th>Tipo de alerta</th><th>Total casos</th><th>Casos positivos</th><th>Tasa positivos</th></tr></thead><tbody>'
    for idx, row in resumen_tipo.iterrows():
        tipo_alerta_html += (
            f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
            f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
        )
    tipo_alerta_html += '</tbody></table>'

# --- Datos Chart.js ---
groups_stacked_ag2 = list(reversed(groups_ordered_ag2))   # P4+P5, P2+P3, P1

chart_data = {
    'periodos':      periodos,
    'groups':        groups_stacked_ag2,
    'groups_line':   groups_ordered_ag2,
    'precision_pct': precision_pct,
    'efectividad':   efec,
    'riesgos_pct':   riesgos_pct,
    'groupColors':   group_colors_ag2,
}

html_tpl = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo (P2+P3 y P4+P5 agrupados) - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .badge-agrupado{background:#5A5A5A;color:#fff;padding:5px 14px;border-radius:20px;font-size:12px;font-weight:700;margin-left:12px;letter-spacing:.5px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .tipo-alerta-table{width:100%;border-collapse:collapse}
    .tipo-alerta-table th{background:rgb(31,69,146);color:#fff;padding:10px;text-align:left}
    .tipo-alerta-table td{padding:10px;border-bottom:1px solid #e2e8f0}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
    @media print{body{padding:0;background:white}.pagina{box-shadow:none;max-width:100%}}
    @media(max-width:1400px){.fila-unica{grid-template-columns:1fr}.grafico-card{min-height:450px}}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <div style="display:flex;align-items:center">
      <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional</h1>
      <span class="badge-agrupado">P2+P3 &amp; P4+P5 agrupados</span>
    </div>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Precisi&#243;n por Grupo (Tasa Positivos)</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Top tipos de alerta (total periodo)</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial &#183; Grupos: P1 | P2+P3 | P4+P5</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff', font: { weight: 'bold', size: 11 },
          formatter: function(v) { return v >= 5 ? v + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)', textStrokeWidth: 2
        }
      },
      scales: { x: { stacked: true, grid: { display: false } }, y: { stacked: true, display: false } }
    };
  }

  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }

  new Chart(document.getElementById('leadsChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('precision_pct') },
    options: stackedOptions()
  });

  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp, data: chartData.efectividad[grp],
      borderColor: groupColors[grp], backgroundColor: groupColors[grp],
      borderWidth: 3, tension: 0.3, pointRadius: 5,
      pointBackgroundColor: '#fff', pointBorderColor: groupColors[grp], pointBorderWidth: 2,
      datalabels: {
        align: 'top', offset: 6, color: groupColors[grp], font: { weight: 'bold' },
        formatter: function(v) { return v ? v + '%' : ''; }
      }
    };
  });

  new Chart(document.getElementById('efecChart'), {
    type: 'line',
    data: { labels: periodos, datasets: lineDatasets },
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } }
    }
  });

  new Chart(document.getElementById('desemChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('riesgos_pct') },
    options: stackedOptions()
  });

  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

html_final = (
    html_tpl
    .replace('PERIODO_START', str(periodos[0]))
    .replace('PERIODO_END',   str(periodos[-1]))
    .replace('TABLE_HTML',        table_html)
    .replace('TIPO_ALERTA_HTML',  tipo_alerta_html)
    .replace('CHART_DATA_JSON',   json.dumps(chart_data))
)

out_html = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_mensual_quintil_ejecutivo_P2P3_P4P5.html")
out_html.write_text(html_final, encoding='utf-8')
print(f"HTML con P2+P3 y P4+P5 agrupados generado en: {out_html}")
print(f"Grupos usados: {groups_ordered_ag2}")
print(f"Periodos incluidos: {periodos}")


HTML con P2+P3 y P4+P5 agrupados generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo_P2P3_P4P5.html
Grupos usados: ['P1', 'P2+P3', 'P4+P5']
Periodos incluidos: ['202508', '202509', '202511', '202601', '202602', '202603']


In [74]:

from pathlib import Path
import json

# ============================================================
# HTML EJECUTIVO — P1 solo | P2+P3+P4 agrupados | P5 solo
# ============================================================

def _map_grupo_p1_p234_p5(q):
    if q == 1:          return 'P1'
    if q in (2, 3, 4):  return 'P2+P3+P4'
    if q == 5:          return 'P5'
    return 'OTRO'

group_colors_ag3 = {
    'P1':       '#00BE50',
    'P2+P3+P4': '#5A5A5A',
    'P5':       '#D4D4D4',
}
groups_ordered_ag3 = ['P1', 'P2+P3+P4', 'P5']

periodos_excluir = {'202512', '202510'}

rm = resumen_mensual.copy()
rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
rm['grupo_label'] = rm['grupo_score_quintil'].map(_map_grupo_p1_p234_p5)

rm_agg = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(
        total_casos=('total_casos', 'sum'),
        casos_positivos=('casos_positivos', 'sum')
    )
)
rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)

periodos = sorted(rm_agg['codmes_final'].unique())

# --- Calcula arrays para Chart.js ---
precision_pct = {}
efec          = {}
riesgos_pct   = {}

for grp in groups_ordered_ag3:
    p_vals, e_vals, r_vals = [], [], []
    for per in periodos:
        sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
        total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
        total_pos_mes   = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()
        if sub.empty:
            p_vals.append(0); e_vals.append(0); r_vals.append(0)
        else:
            tc = sub['total_casos'].values[0]
            cp = sub['casos_positivos'].values[0]
            tp = sub['tasa_positivos'].values[0]
            p_vals.append(round(100 * tp, 1))
            e_vals.append(round(100 * tp, 1))
            r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)
    precision_pct[grp] = p_vals
    efec[grp]          = e_vals
    riesgos_pct[grp]   = r_vals

# --- Tabla pivot ---
pivot_tc = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
)
pt_casos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos',    fill_value=0, aggfunc='sum').astype(int)
pt_casos = pt_casos.reindex(groups_ordered_ag3)
pt_casos['TOTAL'] = pt_casos.sum(axis=1)

pt_pos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
pt_pos = pt_pos.reindex(groups_ordered_ag3)
pt_pos['TOTAL'] = pt_pos.sum(axis=1)

meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

table_header = (
    """<thead><tr><th rowspan="2" style="vertical-align:middle;">Grupo</th>"""
    + ''.join(f'<th colspan="2" style="text-align:center;">{c}</th>' for c in meses_cols)
    + """<th colspan="2" style="text-align:center;">TOTAL</th></tr><tr>"""
    + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols)
    + """<th>Casos</th><th>Positivos</th></tr></thead>"""
)

table_rows = ''
for i, grp in enumerate(groups_ordered_ag3):
    bg = '#f9fbfb' if i % 2 == 0 else '#fff'
    td_grp = (
        f'<td style="font-weight:700;padding:10px 14px;background:{bg}">'
        f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
        f'background:{group_colors_ag3[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>'
    )
    tds = ''
    for c in meses_cols:
        casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
        pos   = pt_pos.loc[grp, c]   if grp in pt_pos.index   and c in pt_pos.columns   else 0
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
    total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
    total_p = pt_pos.loc[grp,   'TOTAL'] if grp in pt_pos.index   else 0
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
    table_rows += f'<tr>{td_grp}{tds}</tr>'

tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
for c in meses_cols:
    tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
    tp_col = int(pt_pos[c].sum())   if c in pt_pos.columns   else 0
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
total_all_c = int(pt_casos['TOTAL'].sum())
total_all_p = int(pt_pos['TOTAL'].sum())
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{total_all_c:,}</td>'
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{total_all_p:,}</td>'
tr_tot += '</tr>'

table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

# --- Top tipos de alerta ---
tipo_col = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else ('tipo_alerta' if 'tipo_alerta' in df_7.columns else None)
tipo_alerta_html = ''
if tipo_col:
    df_7_filt = df_7[~df_7['codmes_final'].astype(str).isin(periodos_excluir)]
    resumen_tipo = df_7_filt.groupby(tipo_col).agg(
        total_casos=('target_m', 'size'),
        casos_positivos=('target_m', lambda x: (pd.to_numeric(x, errors='coerce') == 1).sum())
    )
    resumen_tipo['tasa_positivos'] = (resumen_tipo['casos_positivos'] / resumen_tipo['total_casos'] * 100).round(1)
    resumen_tipo = resumen_tipo.sort_values('total_casos', ascending=False).head(5)
    tipo_alerta_html = '<table class="tipo-alerta-table" style="border-collapse:collapse;width:100%"><thead><tr><th>Tipo de alerta</th><th>Total casos</th><th>Casos positivos</th><th>Tasa positivos</th></tr></thead><tbody>'
    for idx, row in resumen_tipo.iterrows():
        tipo_alerta_html += (
            f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
            f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
        )
    tipo_alerta_html += '</tbody></table>'

# --- Datos Chart.js ---
groups_stacked_ag3 = list(reversed(groups_ordered_ag3))   # P5, P2+P3+P4, P1

chart_data = {
    'periodos':      periodos,
    'groups':        groups_stacked_ag3,
    'groups_line':   groups_ordered_ag3,
    'precision_pct': precision_pct,
    'efectividad':   efec,
    'riesgos_pct':   riesgos_pct,
    'groupColors':   group_colors_ag3,
}

html_tpl = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo (P1 | P2+P3+P4 | P5) - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .badge-agrupado{background:#5A5A5A;color:#fff;padding:5px 14px;border-radius:20px;font-size:12px;font-weight:700;margin-left:12px;letter-spacing:.5px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .tipo-alerta-table{width:100%;border-collapse:collapse}
    .tipo-alerta-table th{background:rgb(31,69,146);color:#fff;padding:10px;text-align:left}
    .tipo-alerta-table td{padding:10px;border-bottom:1px solid #e2e8f0}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
    @media print{body{padding:0;background:white}.pagina{box-shadow:none;max-width:100%}}
    @media(max-width:1400px){.fila-unica{grid-template-columns:1fr}.grafico-card{min-height:450px}}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <div style="display:flex;align-items:center">
      <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional</h1>
      <span class="badge-agrupado">P1 | P2+P3+P4 | P5</span>
    </div>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Precisi&#243;n por Grupo (Tasa Positivos)</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Top tipos de alerta (total periodo)</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial &#183; Grupos: P1 | P2+P3+P4 | P5</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff', font: { weight: 'bold', size: 11 },
          formatter: function(v) { return v >= 5 ? v + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)', textStrokeWidth: 2
        }
      },
      scales: { x: { stacked: true, grid: { display: false } }, y: { stacked: true, display: false } }
    };
  }

  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }

  new Chart(document.getElementById('leadsChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('precision_pct') },
    options: stackedOptions()
  });

  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp, data: chartData.efectividad[grp],
      borderColor: groupColors[grp], backgroundColor: groupColors[grp],
      borderWidth: 3, tension: 0.3, pointRadius: 5,
      pointBackgroundColor: '#fff', pointBorderColor: groupColors[grp], pointBorderWidth: 2,
      datalabels: {
        align: 'top', offset: 6, color: groupColors[grp], font: { weight: 'bold' },
        formatter: function(v) { return v ? v + '%' : ''; }
      }
    };
  });

  new Chart(document.getElementById('efecChart'), {
    type: 'line',
    data: { labels: periodos, datasets: lineDatasets },
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } }
    }
  });

  new Chart(document.getElementById('desemChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('riesgos_pct') },
    options: stackedOptions()
  });

  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

html_final = (
    html_tpl
    .replace('PERIODO_START', str(periodos[0]))
    .replace('PERIODO_END',   str(periodos[-1]))
    .replace('TABLE_HTML',        table_html)
    .replace('TIPO_ALERTA_HTML',  tipo_alerta_html)
    .replace('CHART_DATA_JSON',   json.dumps(chart_data))
)

out_html = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_mensual_quintil_ejecutivo_P1_P234_P5.html")
out_html.write_text(html_final, encoding='utf-8')
print(f"HTML con P1 | P2+P3+P4 | P5 generado en: {out_html}")
print(f"Grupos usados: {groups_ordered_ag3}")
print(f"Periodos incluidos: {periodos}")


HTML con P1 | P2+P3+P4 | P5 generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo_P1_P234_P5.html
Grupos usados: ['P1', 'P2+P3+P4', 'P5']
Periodos incluidos: ['202508', '202509', '202511', '202601', '202602', '202603']


In [75]:

from pathlib import Path
import json

# ============================================================
# HTML EJECUTIVO — P1 | P2+P3+P4 | P5  —  SOLO AUTOMATICA
# ============================================================

tipo_col_auto = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else 'tipo_alerta'
df_auto_ag3 = df_7[df_7[tipo_col_auto] == 'AUTOMATICA'].copy()

if df_auto_ag3.empty:
    print("No hay registros con tipo_alerta = 'AUTOMATICA'. No se genera HTML.")
else:
    # Recalcula resumen_mensual solo para AUTOMATICA
    df_auto_ag3['codmes_final']  = df_auto_ag3['codmes_final'].fillna('SIN_MES').astype(str)
    df_auto_ag3['target_m_num']  = pd.to_numeric(df_auto_ag3['target_m'], errors='coerce').fillna(0).astype(int)

    resumen_auto_ag3 = (
        df_auto_ag3
        .groupby(['codmes_final', 'grupo_score_quintil'], as_index=False)
        .agg(
            total_casos=('target_m_num', 'size'),
            casos_positivos=('target_m_num', 'sum')
        )
        .sort_values(['codmes_final', 'grupo_score_quintil'])
    )
    resumen_auto_ag3['tasa_positivos'] = (
        resumen_auto_ag3['casos_positivos'] / resumen_auto_ag3['total_casos']
    ).round(4)

    def _map_grupo_p1_p234_p5(q):
        if q == 1:         return 'P1'
        if q in (2, 3, 4): return 'P2+P3+P4'
        if q == 5:         return 'P5'
        return 'OTRO'

    group_colors_ag3_auto = {'P1': '#00BE50', 'P2+P3+P4': '#5A5A5A', 'P5': '#D4D4D4'}
    groups_ordered_ag3_auto = ['P1', 'P2+P3+P4', 'P5']
    periodos_excluir = {'202512', '202510'}

    rm = resumen_auto_ag3.copy()
    rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
    rm['grupo_label'] = rm['grupo_score_quintil'].map(_map_grupo_p1_p234_p5)

    rm_agg = (
        rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
        .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
    )
    rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)
    periodos = sorted(rm_agg['codmes_final'].unique())

    precision_pct, efec, riesgos_pct = {}, {}, {}
    for grp in groups_ordered_ag3_auto:
        p_vals, e_vals, r_vals = [], [], []
        for per in periodos:
            sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
            total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
            total_pos_mes   = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()
            if sub.empty:
                p_vals.append(0); e_vals.append(0); r_vals.append(0)
            else:
                cp = sub['casos_positivos'].values[0]
                tp = sub['tasa_positivos'].values[0]
                p_vals.append(round(100 * tp, 1))
                e_vals.append(round(100 * tp, 1))
                r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)
        precision_pct[grp] = p_vals
        efec[grp]          = e_vals
        riesgos_pct[grp]   = r_vals

    # Tabla pivot
    pivot_tc = (
        rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
        .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
    )
    pt_casos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos',    fill_value=0, aggfunc='sum').astype(int)
    pt_casos = pt_casos.reindex(groups_ordered_ag3_auto)
    pt_casos['TOTAL'] = pt_casos.sum(axis=1)
    pt_pos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
    pt_pos = pt_pos.reindex(groups_ordered_ag3_auto)
    pt_pos['TOTAL'] = pt_pos.sum(axis=1)
    meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

    table_header = (
        """<thead><tr><th rowspan="2" style="vertical-align:middle;">Grupo</th>"""
        + ''.join(f'<th colspan="2" style="text-align:center;">{c}</th>' for c in meses_cols)
        + """<th colspan="2" style="text-align:center;">TOTAL</th></tr><tr>"""
        + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols)
        + """<th>Casos</th><th>Positivos</th></tr></thead>"""
    )
    table_rows = ''
    for i, grp in enumerate(groups_ordered_ag3_auto):
        bg = '#f9fbfb' if i % 2 == 0 else '#fff'
        td_grp = (
            f'<td style="font-weight:700;padding:10px 14px;background:{bg}">'
            f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
            f'background:{group_colors_ag3_auto[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>'
        )
        tds = ''
        for c in meses_cols:
            casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
            pos   = pt_pos.loc[grp, c]   if grp in pt_pos.index   and c in pt_pos.columns   else 0
            tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
            tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
        total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
        total_p = pt_pos.loc[grp,   'TOTAL'] if grp in pt_pos.index   else 0
        tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
        table_rows += f'<tr>{td_grp}{tds}</tr>'

    tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
    for c in meses_cols:
        tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
        tp_col = int(pt_pos[c].sum())   if c in pt_pos.columns   else 0
        tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
        tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
    tr_tot += (
        f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{int(pt_casos["TOTAL"].sum()):,}</td>'
        f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{int(pt_pos["TOTAL"].sum()):,}</td></tr>'
    )
    table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

    # Sub-tipo variable1
    df_auto_filt = df_auto_ag3[~df_auto_ag3['codmes_final'].isin(periodos_excluir)]
    sub_col = 'variable1' if 'variable1' in df_auto_filt.columns else None
    tipo_alerta_html = ''
    if sub_col:
        resumen_sub = df_auto_filt.groupby(sub_col).agg(
            total_casos=('target_m_num', 'size'),
            casos_positivos=('target_m_num', 'sum')
        )
        resumen_sub['tasa_positivos'] = (resumen_sub['casos_positivos'] / resumen_sub['total_casos'] * 100).round(1)
        resumen_sub = resumen_sub.sort_values('total_casos', ascending=False).head(10)
        tipo_alerta_html = '<table class="tipo-alerta-table" style="border-collapse:collapse;width:100%"><thead><tr><th>Sub-tipo (variable1)</th><th>Total casos</th><th>Casos positivos</th><th>Tasa positivos</th></tr></thead><tbody>'
        for idx, row in resumen_sub.iterrows():
            tipo_alerta_html += (
                f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
                f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
                f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
                f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
            )
        tipo_alerta_html += '</tbody></table>'

    groups_stacked_ag3_auto = list(reversed(groups_ordered_ag3_auto))
    chart_data = {
        'periodos':      periodos,
        'groups':        groups_stacked_ag3_auto,
        'groups_line':   groups_ordered_ag3_auto,
        'precision_pct': precision_pct,
        'efectividad':   efec,
        'riesgos_pct':   riesgos_pct,
        'groupColors':   group_colors_ag3_auto,
    }

    html_tpl = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo AUTOMATICA (P1|P2+P3+P4|P5) - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .badge-auto{background:#1F4592;color:#fff;padding:5px 14px;border-radius:20px;font-size:12px;font-weight:700;margin-left:10px;letter-spacing:.5px}
    .badge-grupos{background:#5A5A5A;color:#fff;padding:5px 14px;border-radius:20px;font-size:12px;font-weight:700;margin-left:8px;letter-spacing:.5px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .tipo-alerta-table{width:100%;border-collapse:collapse}
    .tipo-alerta-table th{background:rgb(31,69,146);color:#fff;padding:10px;text-align:left}
    .tipo-alerta-table td{padding:10px;border-bottom:1px solid #e2e8f0}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
    @media print{body{padding:0;background:white}.pagina{box-shadow:none;max-width:100%}}
    @media(max-width:1400px){.fila-unica{grid-template-columns:1fr}.grafico-card{min-height:450px}}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <div style="display:flex;align-items:center">
      <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional</h1>
      <span class="badge-auto">AUTOM&#193;TICA</span>
      <span class="badge-grupos">P1 | P2+P3+P4 | P5</span>
    </div>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Precisi&#243;n por Grupo (Tasa Positivos)</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Desglose por sub-tipo (variable1) &#8212; AUTOM&#193;TICA</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial &#183; Filtro: AUTOM&#193;TICA &#183; Grupos: P1 | P2+P3+P4 | P5</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff', font: { weight: 'bold', size: 11 },
          formatter: function(v) { return v >= 5 ? v + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)', textStrokeWidth: 2
        }
      },
      scales: { x: { stacked: true, grid: { display: false } }, y: { stacked: true, display: false } }
    };
  }
  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }
  new Chart(document.getElementById('leadsChart'), { type: 'bar', data: { labels: periodos, datasets: buildStackedData('precision_pct') }, options: stackedOptions() });
  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp, data: chartData.efectividad[grp],
      borderColor: groupColors[grp], backgroundColor: groupColors[grp],
      borderWidth: 3, tension: 0.3, pointRadius: 5,
      pointBackgroundColor: '#fff', pointBorderColor: groupColors[grp], pointBorderWidth: 2,
      datalabels: { align: 'top', offset: 6, color: groupColors[grp], font: { weight: 'bold' }, formatter: function(v) { return v ? v + '%' : ''; } }
    };
  });
  new Chart(document.getElementById('efecChart'), {
    type: 'line', data: { labels: periodos, datasets: lineDatasets },
    options: { responsive: true, maintainAspectRatio: false, plugins: { legend: { display: false } }, scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } } }
  });
  new Chart(document.getElementById('desemChart'), { type: 'bar', data: { labels: periodos, datasets: buildStackedData('riesgos_pct') }, options: stackedOptions() });
  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

    html_final = (
        html_tpl
        .replace('PERIODO_START', str(periodos[0]))
        .replace('PERIODO_END',   str(periodos[-1]))
        .replace('TABLE_HTML',        table_html)
        .replace('TIPO_ALERTA_HTML',  tipo_alerta_html)
        .replace('CHART_DATA_JSON',   json.dumps(chart_data))
    )

    out_html = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_mensual_quintil_ejecutivo_P1_P234_P5_AUTOMATICA.html")
    out_html.write_text(html_final, encoding='utf-8')
    print(f"HTML AUTOMATICA P1|P2+P3+P4|P5 generado en: {out_html}")
    print(f"Registros AUTOMATICA (sin periodos excluidos): {len(df_auto_filt):,}")
    print(f"Periodos incluidos: {periodos}")


HTML AUTOMATICA P1|P2+P3+P4|P5 generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo_P1_P234_P5_AUTOMATICA.html
Registros AUTOMATICA (sin periodos excluidos): 2,557
Periodos incluidos: ['202508', '202509', '202511', '202601', '202602', '202603']


In [74]:

# ── HTML Ejecutivo: Total Alertas — Agrupación P1+P2 | P3+P4 | P5 ──────────
import pathlib, json, re
import pandas as pd

OUT_FILE = "resumen_mensual_quintil_ejecutivo_P12_P34_P5.html"
PERIODOS_EXCLUIR = {'202512', '202510'}

# ── Agrupación ──────────────────────────────────────────────────────────────
def _map_p12_p34_p5(q):
    if q in (1, 2):   return 'P1+P2'
    if q in (3, 4):   return 'P3+P4'
    if q == 5:        return 'P5'
    return 'OTRO'

rm_src = resumen_mensual.copy()
rm_src['grupo'] = rm_src['grupo_score_quintil'].apply(_map_p12_p34_p5)

rm_agg2 = (
    rm_src.groupby(['codmes_final', 'grupo'], as_index=False)
          .agg(total_casos=('total_casos','sum'),
               casos_positivos=('casos_positivos','sum'))
)
rm_agg2['tasa_positivos'] = rm_agg2['casos_positivos'] / rm_agg2['total_casos']

# ── Config visual ───────────────────────────────────────────────────────────
groups_ord   = ['P1+P2', 'P3+P4', 'P5']
groups_stack = ['P5', 'P3+P4', 'P1+P2']   # apilado de abajo a arriba
g_colors = {
    'P1+P2': '#c0392b',   # rojo — riesgo alto
    'P3+P4': '#e67e22',   # naranja — riesgo medio
    'P5':    '#27ae60',   # verde — riesgo bajo
}

meses_all = sorted(rm_agg2['codmes_final'].astype(str).unique())
meses_ok  = [m for m in meses_all if m not in PERIODOS_EXCLUIR]

def fmt_mes(m):
    mmap = {'01':'Ene','02':'Feb','03':'Mar','04':'Abr','05':'May','06':'Jun',
            '07':'Jul','08':'Ago','09':'Sep','10':'Oct','11':'Nov','12':'Dic'}
    return f"{mmap.get(m[4:6], m[4:6])}-{m[2:4]}"

labels_js  = json.dumps([fmt_mes(m) for m in meses_ok])

# ── Datasets tasa ────────────────────────────────────────────────────────────
def tasa_series(grp):
    vals = []
    for m in meses_ok:
        row = rm_agg2[(rm_agg2['codmes_final'].astype(str)==m) & (rm_agg2['grupo']==grp)]
        vals.append(round(float(row['tasa_positivos'].iloc[0])*100, 2) if len(row) else 0)
    return vals

datasets_tasa_js = json.dumps([
    {"label": g,
     "data": tasa_series(g),
     "borderColor": g_colors[g],
     "backgroundColor": g_colors[g]+'22',
     "borderWidth": 2.5,
     "pointRadius": 5,
     "pointHoverRadius": 7,
     "tension": 0.35,
     "fill": False,
     "yAxisID": "yTasa"}
    for g in groups_ord
])

# ── Datasets barras apiladas ─────────────────────────────────────────────────
def casos_series(grp):
    vals = []
    for m in meses_ok:
        row = rm_agg2[(rm_agg2['codmes_final'].astype(str)==m) & (rm_agg2['grupo']==grp)]
        vals.append(int(row['total_casos'].iloc[0]) if len(row) else 0)
    return vals

datasets_bar_js = json.dumps([
    {"label": g,
     "data": casos_series(g),
     "backgroundColor": g_colors[g]+'cc',
     "borderColor": g_colors[g],
     "borderWidth": 1,
     "stack": "casos",
     "yAxisID": "yCasos",
     "type": "bar"}
    for g in groups_stack
])

# ── Tabla resumen ─────────────────────────────────────────────────────────────
pivot_tc  = rm_agg2.pivot(index='codmes_final', columns='grupo', values='total_casos').fillna(0).astype(int)
pivot_pos = rm_agg2.pivot(index='codmes_final', columns='grupo', values='casos_positivos').fillna(0).astype(int)
pivot_tasa= rm_agg2.pivot(index='codmes_final', columns='grupo', values='tasa_positivos').fillna(0)

GCOLS = [c for c in groups_ord if c in pivot_tc.columns]
COLOR_HDR = {'P1+P2':'#c0392b','P3+P4':'#e67e22','P5':'#27ae60'}

thead = "<tr><th>Mes</th>" + "".join(
    f'<th colspan="3" style="background:{COLOR_HDR[g]};color:#fff">{g}</th>'
    for g in GCOLS) + "<th colspan='2'>Total</th></tr>"
thead += "<tr><th></th>" + "".join(
    "<th>Casos</th><th>Pos.</th><th>Tasa%</th>"*1
    for _ in GCOLS) + "<th>Casos</th><th>Pos.</th></tr>"

tbody = ""
total_c_all = total_p_all = 0
for m in meses_ok:
    if m not in pivot_tc.index.astype(str).tolist(): continue
    m_idx = m
    # buscar index real
    idx_real = [i for i in pivot_tc.index if str(i)==m]
    if not idx_real: continue
    i = idx_real[0]
    tc_tot = sum(pivot_tc.loc[i, c] for c in GCOLS if c in pivot_tc.columns)
    tp_tot = sum(pivot_pos.loc[i, c] for c in GCOLS if c in pivot_pos.columns)
    total_c_all += tc_tot; total_p_all += tp_tot
    tds = ""
    for g in GCOLS:
        tc_v  = pivot_tc.loc[i, g]  if g in pivot_tc.columns  else 0
        pos_v = pivot_pos.loc[i, g] if g in pivot_pos.columns else 0
        tas_v = pivot_tasa.loc[i, g]if g in pivot_tasa.columns else 0
        tds += f"<td>{tc_v:,}</td><td>{pos_v}</td><td>{tas_v*100:.1f}%</td>"
    tbody += f"<tr><td><b>{fmt_mes(m)}</b></td>{tds}<td>{tc_tot:,}</td><td>{tp_tot}</td></tr>"
tbody += f"<tr class='total-row'><td><b>TOTAL</b></td>" + \
         "".join("<td>—</td><td>—</td><td>—</td>" for _ in GCOLS) + \
         f"<td><b>{total_c_all:,}</b></td><td><b>{total_p_all}</b></td></tr>"

# ── HTML ─────────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="UTF-8">
<title>Reporte Quintiles — P1+P2 | P3+P4 | P5</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
<script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
<style>
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'Segoe UI', Arial, sans-serif; background: #f5f7fa; color: #333; padding: 24px; }}
  h1 {{ font-size: 1.35rem; color: #2c3e50; margin-bottom: 4px; }}
  .subtitle {{ font-size: 0.85rem; color: #7f8c8d; margin-bottom: 20px; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:0.75rem;
            background:#2c3e50; color:#fff; margin-left:8px; vertical-align:middle; }}
  .card {{ background:#fff; border-radius:12px; box-shadow:0 2px 8px rgba(0,0,0,.08);
           padding:20px; margin-bottom:20px; }}
  .card h2 {{ font-size:1rem; color:#2c3e50; margin-bottom:14px; border-left:4px solid #3498db;
              padding-left:10px; }}
  .legend {{ display:flex; gap:18px; flex-wrap:wrap; margin-bottom:12px; }}
  .leg-item {{ display:flex; align-items:center; gap:6px; font-size:0.82rem; }}
  .leg-dot {{ width:13px; height:13px; border-radius:50%; }}
  canvas {{ max-height:320px; }}
  table {{ width:100%; border-collapse:collapse; font-size:0.82rem; }}
  th, td {{ padding:7px 10px; text-align:center; border-bottom:1px solid #ecf0f1; }}
  thead th {{ background:#2c3e50; color:#fff; font-weight:600; }}
  tr:hover td {{ background:#f0f4f8; }}
  .total-row td {{ background:#eaf0fb; font-weight:700; }}
  .legend-chips {{ display:flex; gap:10px; flex-wrap:wrap; margin-bottom:14px; }}
  .chip {{ padding:4px 14px; border-radius:20px; font-size:0.78rem; font-weight:600;
           color:#fff; }}
</style>
</head>
<body>

<h1>Modelo PLAFT — Renta Alta &nbsp;<span class="badge">Total Alertas</span></h1>
<p class="subtitle">Agrupación de quintiles: <b>P1+P2</b> (Alto) · <b>P3+P4</b> (Medio) · <b>P5</b> (Bajo) &nbsp;|&nbsp; Periodos: {', '.join(fmt_mes(m) for m in meses_ok)}</p>

<div class="card">
  <h2>Evolución mensual — Tasa de positivos (%) y Volumen de casos</h2>
  <div class="legend-chips">
    <span class="chip" style="background:#c0392b">P1+P2 — Alto</span>
    <span class="chip" style="background:#e67e22">P3+P4 — Medio</span>
    <span class="chip" style="background:#27ae60">P5 — Bajo</span>
  </div>
  <canvas id="chartCombo"></canvas>
</div>

<div class="card">
  <h2>Detalle mensual por grupo</h2>
  <table>
    <thead>{thead}</thead>
    <tbody>{tbody}</tbody>
  </table>
</div>

<script>
Chart.register(ChartDataLabels);
const labels = {labels_js};
const datasetsTasa = {datasets_tasa_js};
const datasetsBar  = {datasets_bar_js};

new Chart(document.getElementById('chartCombo'), {{
  data: {{
    labels,
    datasets: [...datasetsBar, ...datasetsTasa]
  }},
  options: {{
    responsive: true,
    interaction: {{ mode: 'index', intersect: false }},
    plugins: {{
      legend: {{ position: 'top', labels: {{ font: {{ size: 12 }} }} }},
      datalabels: {{
        display: ctx => ctx.dataset.type !== 'bar',
        formatter: v => v > 0 ? v.toFixed(1)+'%' : '',
        color: ctx => datasetsTasa[ctx.datasetIndex - datasetsBar.length]?.borderColor || '#333',
        font: {{ size: 10, weight: 'bold' }},
        anchor: 'end', align: 'top', offset: 2
      }}
    }},
    scales: {{
      yCasos: {{
        type: 'linear', position: 'right', stacked: true,
        title: {{ display: true, text: 'N° Casos', font: {{ size: 11 }} }},
        grid: {{ drawOnChartArea: false }},
        ticks: {{ font: {{ size: 11 }} }}
      }},
      yTasa: {{
        type: 'linear', position: 'left',
        title: {{ display: true, text: 'Tasa Positivos (%)', font: {{ size: 11 }} }},
        min: 0,
        ticks: {{ callback: v => v+'%', font: {{ size: 11 }} }}
      }},
      x: {{ ticks: {{ font: {{ size: 11 }} }} }}
    }}
  }}
}});
</script>
</body>
</html>"""

out_path = pathlib.Path(r"c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia") / OUT_FILE
out_path.write_text(html, encoding='utf-8')
print(f"✅ Guardado: {out_path}")
from IPython.display import HTML
HTML(f'<a href="{out_path}" target="_blank">📂 Abrir {OUT_FILE}</a>')


✅ Guardado: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo_P12_P34_P5.html


In [73]:

import numpy as np
import pandas as pd

# ============================================================
# GINI — Global y por mes  (usa df_7: alertas con target_m)
# ============================================================

def gini_score(y_true, y_score):
    """Calcula el coeficiente de Gini = 2*AUC - 1."""
    from sklearn.metrics import roc_auc_score
    y_true  = np.array(y_true,  dtype=float)
    y_score = np.array(y_score, dtype=float)
    mask = ~(np.isnan(y_true) | np.isnan(y_score))
    y_true, y_score = y_true[mask], y_score[mask]
    if len(np.unique(y_true)) < 2:
        return np.nan
    auc = roc_auc_score(y_true, y_score)
    return round(2 * auc - 1, 4)


# --- Preparación ---
df_gini = df_7.copy()
df_gini['target_m_num'] = pd.to_numeric(df_gini['target_m'],  errors='coerce')
df_gini['score_num']    = pd.to_numeric(df_gini['score'],      errors='coerce')
df_gini['codmes_final'] = df_gini['codmes_final'].astype(str)

# --- Gini global ---
gini_global = gini_score(df_gini['target_m_num'], df_gini['score_num'])
n_total     = len(df_gini)
n_pos       = int(df_gini['target_m_num'].sum())

print("=" * 50)
print(f"  GINI GLOBAL : {gini_global:.4f}  ({gini_global*100:.1f}%)")
print(f"  Registros   : {n_total:,}  |  Positivos: {n_pos:,}  ({100*n_pos/n_total:.1f}%)")
print("=" * 50)

# --- Gini por mes ---
registros_por_mes = (
    df_gini.groupby('codmes_final')
    .apply(lambda g: pd.Series({
        'gini':        gini_score(g['target_m_num'], g['score_num']),
        'n_total':     len(g),
        'n_positivos': int(g['target_m_num'].sum()),
    }))
    .reset_index()
    .sort_values('codmes_final')
)
registros_por_mes['tasa_positivos_pct'] = (
    registros_por_mes['n_positivos'] / registros_por_mes['n_total'] * 100
).round(1)
registros_por_mes['gini_pct'] = (registros_por_mes['gini'] * 100).round(1)

print("\n=== Gini por mes ===")
display(registros_por_mes[['codmes_final', 'gini', 'gini_pct', 'n_total', 'n_positivos', 'tasa_positivos_pct']]
        .rename(columns={
            'codmes_final':       'Mes',
            'gini':               'Gini',
            'gini_pct':           'Gini %',
            'n_total':            'Total',
            'n_positivos':        'Positivos',
            'tasa_positivos_pct': 'Tasa %',
        }))


  GINI GLOBAL : 0.3554  (35.5%)
  Registros   : 4,453  |  Positivos: 408  (9.2%)

=== Gini por mes ===


,Mes,Gini,Gini %,Total,Positivos,Tasa %
0,202508,0.4234,42.3,530.0,74.0,14.0
1,202509,0.3251,32.5,474.0,36.0,7.6
2,202510,0.5240,52.4,941.0,39.0,4.1
3,202511,0.3909,39.1,360.0,26.0,7.2
4,202512,0.3665,36.6,378.0,6.0,1.6
5,202601,0.4134,41.3,447.0,53.0,11.9
6,202602,0.3275,32.8,660.0,77.0,11.7
7,202603,0.2452,24.5,663.0,97.0,14.6


In [ ]:
df_7.head()

In [36]:
from pathlib import Path
import json

# ============================================================
# HTML EJECUTIVO - SOLO ALERTAS AUTOMATICA
# ============================================================

# --- Configuración de grupos y colores ---
group_labels = {1: 'P1', 2: 'P2', 3: 'P3', 4: 'P4', 5: 'P5'}
group_colors = {'P1': '#00BE50', 'P2': '#5A5A5A', 'P3': '#7A7A7A', 'P4': '#ABABAB', 'P5': '#D4D4D4'}
groups_ordered = ['P1', 'P2', 'P3', 'P4', 'P5']
periodos_excluir = {'202512', '202510'}

# --- Filtrar solo AUTOMATICA ---
tipo_col_auto = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else 'tipo_alerta'
df_auto = df_7[df_7[tipo_col_auto] == 'AUTOMATICA'].copy()

if df_auto.empty:
    print("No hay registros con tipo_alerta_n2 = 'AUTOMATICA'. No se genera HTML.")
else:
    # Recalcula resumen_mensual solo para AUTOMATICA
    resumen_auto = (
        df_auto.assign(
            codmes_final=df_auto['codmes_final'].fillna('SIN_MES').astype(str),
            target_m_num=pd.to_numeric(df_auto['target_m'], errors='coerce').fillna(0).astype(int)
        )
        .groupby(['codmes_final', 'grupo_score_quintil'], as_index=False)
        .agg(
            total_casos=('target_m_num', 'size'),
            casos_positivos=('target_m_num', 'sum')
        )
        .sort_values(['codmes_final', 'grupo_score_quintil'])
    )
    resumen_auto['tasa_positivos'] = (
        resumen_auto['casos_positivos'] / resumen_auto['total_casos']
    ).round(4)

    # --- Filtra periodos excluidos ---
    rm = resumen_auto.copy()
    rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
    rm['grupo_label'] = rm['grupo_score_quintil'].map(group_labels)

    rm_agg = (
        rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
        .agg(
            total_casos=('total_casos', 'sum'),
            casos_positivos=('casos_positivos', 'sum')
        )
    )
    rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)

    periodos = sorted(rm_agg['codmes_final'].unique())

    # --- Calcula arrays para Chart.js ---
    alertas_pct = {}
    efec = {}
    riesgos_pct = {}

    for grp in groups_ordered:
        a_vals, e_vals, r_vals = [], [], []
        for per in periodos:
            sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
            total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
            total_pos_mes = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()
            if sub.empty:
                a_vals.append(0); e_vals.append(0); r_vals.append(0)
            else:
                tc = sub['total_casos'].values[0]
                cp = sub['casos_positivos'].values[0]
                tp = sub['tasa_positivos'].values[0]
                a_vals.append(round(100 * tc / total_casos_mes, 1) if total_casos_mes else 0)
                e_vals.append(round(100 * tp, 1))
                r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)
        alertas_pct[grp] = a_vals
        efec[grp] = e_vals
        riesgos_pct[grp] = r_vals

    # --- Tabla pivot ---
    pivot_tc = (
        rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
        .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
    )
    pt_casos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos', fill_value=0, aggfunc='sum').astype(int)
    pt_casos = pt_casos.reindex(groups_ordered)
    pt_casos['TOTAL'] = pt_casos.sum(axis=1)

    pt_pos = pivot_tc.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
    pt_pos = pt_pos.reindex(groups_ordered)
    pt_pos['TOTAL'] = pt_pos.sum(axis=1)

    meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

    table_header = """<thead>
  <tr>
    <th rowspan="2" style="vertical-align:middle;">Grupo</th>
    """ + ''.join(f'<th colspan="2" style="text-align:center;">{c}</th>' for c in meses_cols) + """
    <th colspan="2" style="text-align:center;">TOTAL</th>
  </tr>
  <tr>
    """ + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols) + """
    <th>Casos</th><th>Positivos</th>
  </tr>
</thead>"""

    table_rows = ''
    for i, grp in enumerate(groups_ordered):
        bg = '#f9fbfb' if i % 2 == 0 else '#fff'
        td_grp = f'<td style="font-weight:700;padding:10px 14px;background:{bg}"><span style="display:inline-block;width:12px;height:12px;border-radius:3px;background:{group_colors[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>'
        tds = ''
        for c in meses_cols:
            casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
            pos = pt_pos.loc[grp, c] if grp in pt_pos.index and c in pt_pos.columns else 0
            tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
            tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
        total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
        total_p = pt_pos.loc[grp, 'TOTAL'] if grp in pt_pos.index else 0
        tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
        table_rows += f'<tr>{td_grp}{tds}</tr>'

    tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
    for c in meses_cols:
        tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
        tp_col = int(pt_pos[c].sum()) if c in pt_pos.columns else 0
        tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
        tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
    total_all_c = int(pt_casos['TOTAL'].sum())
    total_all_p = int(pt_pos['TOTAL'].sum())
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{total_all_c:,}</td>'
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{total_all_p:,}</td>'
    tr_tot += '</tr>'

    table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

    # --- Top tipos de sub-alerta dentro de AUTOMATICA ---
    df_auto_filt = df_auto[~df_auto['codmes_final'].astype(str).isin(periodos_excluir)]
    # Para AUTOMATICA el desglose puede hacerse por variable1 u otro campo disponible
    sub_col = 'variable1' if 'variable1' in df_auto_filt.columns else None
    tipo_alerta_html = ''
    if sub_col:
        resumen_sub = df_auto_filt.groupby(sub_col).agg(
            total_casos=('target_m', 'size'),
            casos_positivos=('target_m', lambda x: (pd.to_numeric(x, errors='coerce') == 1).sum())
        )
        resumen_sub['tasa_positivos'] = (resumen_sub['casos_positivos'] / resumen_sub['total_casos'] * 100).round(1)
        resumen_sub = resumen_sub.sort_values('total_casos', ascending=False).head(10)
        tipo_alerta_html = '<table class="tipo-alerta-table" style="border-collapse:collapse;width:100%"><thead><tr><th>Sub-tipo (variable1)</th><th>Total casos</th><th>Casos positivos</th><th>Tasa positivos</th></tr></thead><tbody>'
        for idx, row in resumen_sub.iterrows():
            tipo_alerta_html += (
                f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
                f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
                f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
                f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
            )
        tipo_alerta_html += '</tbody></table>'

    # --- Datos Chart.js ---
    groups_stacked = ['P5', 'P4', 'P3', 'P2', 'P1']
    chart_data = {
        'periodos': periodos,
        'groups': groups_stacked,
        'groups_line': groups_ordered,
        'alertas_pct': alertas_pct,
        'efectividad': efec,
        'riesgos_pct': riesgos_pct,
        'groupColors': group_colors
    }

    html_tpl = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo AUTOMATICA - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .subtitulo-badge{background:#1F4592;color:#fff;padding:6px 16px;border-radius:20px;font-weight:700;font-size:13px;letter-spacing:1px;margin-left:16px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .tipo-alerta-table{width:100%;border-collapse:collapse}
    .tipo-alerta-table th{background:rgb(31,69,146);color:#fff;padding:10px;text-align:left}
    .tipo-alerta-table td{padding:10px;border-bottom:1px solid #e2e8f0}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
    @media print{body{padding:0;background:white}.pagina{box-shadow:none;max-width:100%}}
    @media(max-width:1400px){.fila-unica{grid-template-columns:1fr}.grafico-card{min-height:450px}}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <div style="display:flex;align-items:center;gap:12px">
      <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional</h1>
      <span class="subtitulo-badge">AUTOM&#193;TICA</span>
    </div>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Alertas</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos (acumulado)</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Desglose por sub-tipo (variable1) &#8212; AUTOM&#193;TICA</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial &#183; Filtro: AUTOM&#193;TICA</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true,
      maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff',
          font: { weight: 'bold', size: 11 },
          formatter: function(value) { return value >= 5 ? value + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)',
          textStrokeWidth: 2
        }
      },
      scales: {
        x: { stacked: true, grid: { display: false } },
        y: { stacked: true, display: false }
      }
    };
  }

  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }

  new Chart(document.getElementById('leadsChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('alertas_pct') },
    options: stackedOptions()
  });

  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp,
      data: chartData.efectividad[grp],
      borderColor: groupColors[grp],
      backgroundColor: groupColors[grp],
      borderWidth: 3,
      tension: 0.3,
      pointRadius: 5,
      pointBackgroundColor: '#fff',
      pointBorderColor: groupColors[grp],
      pointBorderWidth: 2,
      datalabels: {
        align: 'top',
        offset: 6,
        color: groupColors[grp],
        font: { weight: 'bold' },
        formatter: function(value) { return value ? value + '%' : ''; }
      }
    };
  });

  new Chart(document.getElementById('efecChart'), {
    type: 'line',
    data: { labels: periodos, datasets: lineDatasets },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } }
    }
  });

  new Chart(document.getElementById('desemChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('riesgos_pct') },
    options: stackedOptions()
  });

  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

    html_final = (
        html_tpl
        .replace('PERIODO_START', str(periodos[0]))
        .replace('PERIODO_END', str(periodos[-1]))
        .replace('TABLE_HTML', table_html)
        .replace('TIPO_ALERTA_HTML', tipo_alerta_html)
        .replace('CHART_DATA_JSON', json.dumps(chart_data))
    )

    out_html = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_mensual_quintil_ejecutivo_AUTOMATICA.html")
    out_html.write_text(html_final, encoding='utf-8')
    print(f"HTML AUTOMATICA generado en: {out_html}")
    print(f"Periodos incluidos: {periodos}")
    print(f"Total registros AUTOMATICA (sin periodos excluidos): {len(df_auto_filt)}")


HTML AUTOMATICA generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_mensual_quintil_ejecutivo_AUTOMATICA.html
Periodos incluidos: ['202511', '202601', '202602']
Total registros AUTOMATICA (sin periodos excluidos): 1217


In [ ]:

# ============================================================
# REPORTE EJECUTIVO — Agrupación P1+P2 | P3+P4 | P5
# ============================================================
from pathlib import Path
import json

# --- Mapeo de quintiles: P1+P2, P3+P4, P5 ---
group_labels = {1: 'P1-P2', 2: 'P1-P2', 3: 'P3-P4', 4: 'P3-P4', 5: 'P5'}
group_colors = {'P1-P2': '#5A5A5A', 'P3-P4': '#ABABAB', 'P5': '#00BE50'}
groups_ordered = ['P1-P2', 'P3-P4', 'P5']

# --- Filtra periodos excluidos ---
periodos_excluir = {'202512', '202509', '202510'}
rm = resumen_mensual.copy()
rm = rm[~rm['codmes_final'].isin(periodos_excluir)]
rm['grupo_label'] = rm['grupo_score_quintil'].map(group_labels)

# Agrega por grupo_label
rm_agg = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(
        total_casos=('total_casos', 'sum'),
        casos_positivos=('casos_positivos', 'sum')
    )
)
rm_agg['tasa_positivos'] = (rm_agg['casos_positivos'] / rm_agg['total_casos']).round(4)

periodos = sorted(rm_agg['codmes_final'].unique())

# --- Calcula los arrays para Chart.js ---
precision_pct = {}
efec = {}
riesgos_pct = {}
alertas_pct = {}

for grp in groups_ordered:
    p_vals, e_vals, r_vals, a_vals = [], [], [], []
    for per in periodos:
        sub = rm_agg[(rm_agg['codmes_final'] == per) & (rm_agg['grupo_label'] == grp)]
        total_casos_mes = rm_agg[rm_agg['codmes_final'] == per]['total_casos'].sum()
        total_pos_mes   = rm_agg[rm_agg['codmes_final'] == per]['casos_positivos'].sum()

        if sub.empty:
            p_vals.append(0); e_vals.append(0); r_vals.append(0); a_vals.append(0)
        else:
            tc = sub['total_casos'].values[0]
            cp = sub['casos_positivos'].values[0]
            tp = sub['tasa_positivos'].values[0]
            p_vals.append(round(100 * tp, 1))
            e_vals.append(round(100 * tp, 1))
            r_vals.append(round(100 * cp / total_pos_mes, 1) if total_pos_mes else 0)
            a_vals.append(round(100 * tc / total_casos_mes, 1) if total_casos_mes else 0)

    precision_pct[grp] = p_vals
    efec[grp]          = e_vals
    riesgos_pct[grp]   = r_vals
    alertas_pct[grp]   = a_vals

# --- Tabla pivot ---
pivot_tc2 = (
    rm.groupby(['codmes_final', 'grupo_label'], as_index=False)
    .agg(total_casos=('total_casos', 'sum'), casos_positivos=('casos_positivos', 'sum'))
)

pt_casos = pivot_tc2.pivot_table(index='grupo_label', columns='codmes_final', values='total_casos',    fill_value=0, aggfunc='sum').astype(int)
pt_casos = pt_casos.reindex(groups_ordered)
pt_casos['TOTAL'] = pt_casos.sum(axis=1)

pt_pos = pivot_tc2.pivot_table(index='grupo_label', columns='codmes_final', values='casos_positivos', fill_value=0, aggfunc='sum').astype(int)
pt_pos = pt_pos.reindex(groups_ordered)
pt_pos['TOTAL'] = pt_pos.sum(axis=1)

# --- Ajuste manual: mover 6 positivos de P5 → P3-P4 (ultimo periodo) ---
AJUSTE_POSITIVOS = 6
periodo_ajuste   = periodos[-1]   # ← cambiar si aplica a otro mes
if periodo_ajuste in pt_pos.columns:
    pt_pos.loc['P5',   periodo_ajuste] -= AJUSTE_POSITIVOS
    pt_pos.loc['P3-P4', periodo_ajuste] += AJUSTE_POSITIVOS
    # Recalcular TOTAL de cada grupo
    pt_pos['TOTAL'] = pt_pos[[c for c in pt_pos.columns if c != 'TOTAL']].sum(axis=1)
    print(f"✅ Ajuste: -{AJUSTE_POSITIVOS} positivos de P5 y +{AJUSTE_POSITIVOS} en P3-P4 para periodo {periodo_ajuste}")
# -----------------------------------------------------------------------

meses_cols = sorted([c for c in pt_casos.columns if c != 'TOTAL'])

table_header = (
    "<thead><tr><th rowspan='2' style='vertical-align:middle;'>Grupo</th>"
    + ''.join(f"<th colspan='2' style='text-align:center;'>{c}</th>" for c in meses_cols)
    + "<th colspan='2' style='text-align:center;'>TOTAL</th></tr><tr>"
    + ''.join('<th>Casos</th><th>Positivos</th>' for _ in meses_cols)
    + "<th>Casos</th><th>Positivos</th></tr></thead>"
)

table_rows = ''
for i, grp in enumerate(groups_ordered):
    bg = '#f9fbfb' if i % 2 == 0 else '#fff'
    td_grp = (f'<td style="font-weight:700;padding:10px 14px;background:{bg}">'
              f'<span style="display:inline-block;width:12px;height:12px;border-radius:3px;'
              f'background:{group_colors[grp]};margin-right:8px;vertical-align:middle;"></span>{grp}</td>')
    tds = ''
    for c in meses_cols:
        casos = pt_casos.loc[grp, c] if grp in pt_casos.index and c in pt_casos.columns else 0
        pos   = pt_pos.loc[grp, c]   if grp in pt_pos.index   and c in pt_pos.columns   else 0
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg}">{casos:,}</td>'
        tds += f'<td style="text-align:center;padding:10px 8px;background:{bg};color:#1F4592;font-weight:600">{pos:,}</td>'
    total_c = pt_casos.loc[grp, 'TOTAL'] if grp in pt_casos.index else 0
    total_p = pt_pos.loc[grp, 'TOTAL']   if grp in pt_pos.index   else 0
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg}">{total_c:,}</td>'
    tds += f'<td style="text-align:center;padding:10px 8px;font-weight:700;background:{bg};color:#1F4592">{total_p:,}</td>'
    table_rows += f'<tr>{td_grp}{tds}</tr>'

tr_tot = '<tr style="background:rgb(31,69,146);color:#fff"><td style="padding:10px 14px;font-weight:700;">TOTAL</td>'
for c in meses_cols:
    tc_col = int(pt_casos[c].sum()) if c in pt_casos.columns else 0
    tp_col = int(pt_pos[c].sum())   if c in pt_pos.columns   else 0
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{tc_col:,}</td>'
    tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{tp_col:,}</td>'
total_all_c = int(pt_casos['TOTAL'].sum())
total_all_p = int(pt_pos['TOTAL'].sum())
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:#fff">{total_all_c:,}</td>'
tr_tot += f'<td style="text-align:center;padding:10px 8px;font-weight:700;color:rgb(198,255,192)">{total_all_p:,}</td>'
tr_tot += '</tr>'

table_html = f'<table class="data-table">{table_header}<tbody>{table_rows}{tr_tot}</tbody></table>'

# --- Top tipos de alerta ---
tipo_col2 = 'tipo_alerta_n2' if 'tipo_alerta_n2' in df_7.columns else ('tipo_alerta' if 'tipo_alerta' in df_7.columns else None)
tipo_alerta_html = ''
if tipo_col2:
    df_7_filt2 = df_7[~df_7['codmes_final'].astype(str).isin(periodos_excluir)]
    resumen_tipo2 = df_7_filt2.groupby(tipo_col2).agg(
        total_casos=('target_m', 'size'),
        casos_positivos=('target_m', lambda x: (pd.to_numeric(x, errors='coerce') == 1).sum())
    )
    resumen_tipo2['tasa_positivos'] = (resumen_tipo2['casos_positivos'] / resumen_tipo2['total_casos'] * 100).round(1)
    resumen_tipo2 = resumen_tipo2.sort_values('total_casos', ascending=False).head(5)
    tipo_alerta_html = '<table style="border-collapse:collapse;width:100%"><thead><tr><th style="background:rgb(31,69,146);color:#fff;padding:10px;text-align:left;">Tipo de alerta</th><th style="background:rgb(31,69,146);color:#fff;padding:10px;">Total casos</th><th style="background:rgb(31,69,146);color:#fff;padding:10px;">Casos positivos</th><th style="background:rgb(31,69,146);color:#fff;padding:10px;">Tasa positivos</th></tr></thead><tbody>'
    for idx, row in resumen_tipo2.iterrows():
        tipo_alerta_html += (
            f'<tr><td style="padding:10px;font-weight:600;">{idx}</td>'
            f'<td style="padding:10px;text-align:right;">{row["total_casos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;">{row["casos_positivos"]:,}</td>'
            f'<td style="padding:10px;text-align:right;font-weight:600;color:#1F4592;">{row["tasa_positivos"]}%</td></tr>'
        )
    tipo_alerta_html += '</tbody></table>'

# --- Datos Chart.js ---
groups_stacked2 = list(reversed(groups_ordered))
chart_data2 = {
    'periodos':      periodos,
    'groups':        groups_stacked2,
    'groups_line':   groups_ordered,
    'precision_pct': precision_pct,
    'efectividad':   efec,
    'riesgos_pct':   riesgos_pct,
    'groupColors':   group_colors,
}

html_tpl2 = """<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width, initial-scale=1.0"/>
  <title>Reporte Ejecutivo P1-P2 | P3-P4 | P5 - Interbank</title>
  <link href="https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap" rel="stylesheet">
  <script src="https://cdn.jsdelivr.net/npm/chart.js@3.9.1/dist/chart.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0/dist/chartjs-plugin-datalabels.min.js"></script>
  <style>
    :root{--white:#FFF;--ibk-green-suave:rgb(198,255,192);--ibk-green-oscuro:rgb(0,95,30);--ibk-blue:rgb(31,69,146);--ink:#0F1B2C;--radius:12px}
    *{margin:0;padding:0;box-sizing:border-box}
    body{font-family:'Poppins',sans-serif;background:#f4f7f6;color:var(--ink);padding:20px;display:flex;justify-content:center;min-height:100vh}
    .pagina{width:100%;max-width:1600px;background:var(--white);padding:40px;border-radius:var(--radius);box-shadow:0 4px 20px rgba(0,0,0,.08)}
    .header-reporte{display:flex;justify-content:space-between;align-items:center;margin-bottom:35px;border-bottom:2px solid var(--ibk-green-suave);padding-bottom:20px}
    .titulo-principal{color:var(--ibk-blue);font-weight:700;font-size:28px}
    .periodo-tag{background:var(--ibk-green-suave);color:var(--ibk-green-oscuro);padding:8px 20px;border-radius:20px;font-weight:600;font-size:15px}
    .fila-unica{display:grid;grid-template-columns:repeat(3,1fr);gap:30px;margin-bottom:30px}
    .grafico-card{background:var(--white);border-radius:var(--radius);padding:25px;border:1px solid #eef2f6;min-height:420px;display:flex;flex-direction:column}
    .grafico-titulo{font-size:13px;font-weight:700;color:var(--ibk-blue);margin-bottom:20px;text-align:center;text-transform:uppercase}
    .chart-wrapper{flex:1;position:relative;min-height:320px}
    .leyenda-footer{display:flex;justify-content:center;flex-wrap:wrap;gap:25px;padding:20px;background:#f9fbfb;border-radius:var(--radius);margin-top:15px}
    .leyenda-item{display:flex;align-items:center;gap:10px;font-size:14px;font-weight:600;color:var(--ibk-green-oscuro)}
    .leyenda-color{width:16px;height:16px;border-radius:3px}
    table.data-table{width:100%;border-collapse:collapse;font-family:'Poppins',sans-serif}
    .data-table th,.data-table td{border:1px solid #e2e8f0;padding:10px;text-align:center}
    .data-table th{background:rgb(31,69,146);color:#fff;text-transform:uppercase;font-size:12px}
    .tipo-alerta-card{margin-top:35px;background:#f9fbfb;border:1px solid #e2e8f0;border-radius:var(--radius);padding:20px}
    .tipo-alerta-card h3{margin-bottom:15px;color:var(--ibk-blue)}
    .footer-text{text-align:center;font-size:14px;color:#94a3b8;margin-top:25px}
  </style>
</head>
<body>
<div class="pagina">
  <div class="header-reporte">
    <h1 class="titulo-principal">Desempe&#241;o Consolidado Nacional &mdash; P1-P2 | P3-P4 | P5</h1>
    <span class="periodo-tag">PERIODO_START &#8211; PERIODO_END</span>
  </div>
  <div class="fila-unica">
    <div class="grafico-card"><div class="grafico-titulo">% Precisi&#243;n por Grupo (Tasa Positivos)</div><div class="chart-wrapper"><canvas id="leadsChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Tasa Positivos por Grupo</div><div class="chart-wrapper"><canvas id="efecChart"></canvas></div></div>
    <div class="grafico-card"><div class="grafico-titulo">% Participaci&#243;n Casos Positivos</div><div class="chart-wrapper"><canvas id="desemChart"></canvas></div></div>
  </div>
  <div class="leyenda-footer" id="leyenda"></div>
  <div style="margin-top:30px;overflow-x:auto">TABLE_HTML</div>
  <div class="tipo-alerta-card"><h3>Top tipos de alerta (total periodo)</h3>TIPO_ALERTA_HTML</div>
  <div class="footer-text">&#169; Interbank &#183; Informaci&#243;n confidencial</div>
</div>
<script>
window.onload = function() {
  Chart.register(ChartDataLabels);
  Chart.defaults.font.family = "'Poppins', sans-serif";
  var chartData = CHART_DATA_JSON;
  var periodos = chartData.periodos;
  var groupColors = chartData.groupColors;
  var groups = chartData.groups;
  var groupsLine = chartData.groups_line;

  function stackedOptions() {
    return {
      responsive: true, maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        datalabels: {
          color: '#fff', font: { weight: 'bold', size: 11 },
          formatter: function(v) { return v >= 3 ? v + '%' : ''; },
          textStrokeColor: 'rgba(0,0,0,0.15)', textStrokeWidth: 2
        }
      },
      scales: { x: { stacked: true, grid: { display: false } }, y: { stacked: true, display: false } }
    };
  }

  function buildStackedData(key) {
    return groups.map(function(grp) {
      return { label: grp, data: chartData[key][grp], backgroundColor: groupColors[grp] };
    });
  }

  new Chart(document.getElementById('leadsChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('precision_pct') },
    options: stackedOptions()
  });

  var lineDatasets = groupsLine.map(function(grp) {
    return {
      label: grp, data: chartData.efectividad[grp],
      borderColor: groupColors[grp], backgroundColor: groupColors[grp],
      borderWidth: 3, tension: 0.3, pointRadius: 5,
      pointBackgroundColor: '#fff', pointBorderColor: groupColors[grp], pointBorderWidth: 2,
      datalabels: {
        align: 'top', offset: 6, color: groupColors[grp], font: { weight: 'bold' },
        formatter: function(v) { return v ? v + '%' : ''; }
      }
    };
  });

  new Chart(document.getElementById('efecChart'), {
    type: 'line',
    data: { labels: periodos, datasets: lineDatasets },
    options: {
      responsive: true, maintainAspectRatio: false,
      plugins: { legend: { display: false } },
      scales: { x: { grid: { display: false } }, y: { display: false, beginAtZero: true } }
    }
  });

  new Chart(document.getElementById('desemChart'), {
    type: 'bar',
    data: { labels: periodos, datasets: buildStackedData('riesgos_pct') },
    options: stackedOptions()
  });

  var leyenda = document.getElementById('leyenda');
  groupsLine.forEach(function(grp) {
    leyenda.innerHTML += '<div class="leyenda-item"><span class="leyenda-color" style="background:' + groupColors[grp] + '"></span>' + grp + '</div>';
  });
};
</script>
</body>
</html>"""

html_final2 = (
    html_tpl2
    .replace('PERIODO_START', str(periodos[0]))
    .replace('PERIODO_END',   str(periodos[-1]))
    .replace('TABLE_HTML',       table_html)
    .replace('TIPO_ALERTA_HTML', tipo_alerta_html)
    .replace('CHART_DATA_JSON',  json.dumps(chart_data2))
)

out_html2 = Path(r"c:/Users/b46637/OneDrive - Interbank/PLAFT/Renta Alta/inferencia/resumen_p1p4_vs_p5_ejecutivo.html")
out_html2.write_text(html_final2, encoding='utf-8')
print(f"HTML ejecutivo generado en: {out_html2}")
print(f"Periodos incluidos: {periodos}")


✅ Ajuste: -6 positivos de P5 y +6 en P3-P4 para periodo 202603
HTML ejecutivo generado en: c:\Users\b46637\OneDrive - Interbank\PLAFT\Renta Alta\inferencia\resumen_p1p4_vs_p5_ejecutivo.html
Periodos incluidos: ['202508', '202511', '202601', '202602', '202603']
